### Note - Run all cells before ------setup complete--------- to setup the database

# 📘 Function Documentation (E-commerce DB Notebook)


This section documents the utility functions used for querying, inspecting, and profiling the MySQL `ecommerce_db` database using SQLAlchemy and Pandas.

---

## 🔹 Function: `show(query)`

**Description:**
Executes a SQL query and returns the result as a Pandas DataFrame. Useful for quickly viewing query outputs inside the notebook.

**Parameters:**

| Name  | Type | Description                     |
| ----- | ---- | ------------------------------- |
| query | str  | SQL query string to be executed |

**Returns:**

| Type             | Description                 |
| ---------------- | --------------------------- |
| pandas.DataFrame | Query result as a DataFrame |

**Example Usage:**

```python
show("SELECT * FROM users LIMIT 5;")
```

**Notes:**

* Assumes a valid database connection via `engine`
* Works for SELECT and metadata queries

---

## 🔹 Function: `explain(query)`

**Description:**
Runs `EXPLAIN ANALYZE` on a given SQL query and prints the execution plan for performance analysis.

**Parameters:**

| Name  | Type | Description          |
| ----- | ---- | -------------------- |
| query | str  | SQL query to analyze |

**Returns:**

| Type | Description                      |
| ---- | -------------------------------- |
| None | Prints execution plan to console |

**Example Usage:**

```python
explain("SELECT * FROM orders WHERE user_id = 1")
```

**Notes:**

* Helps understand query performance and optimization
* Output is printed, not returned

---

## 🔹 Function: `last_query_profile()`

**Description:**
Fetches the profiling details of the most recently executed query using MySQL profiling and returns a breakdown of execution time by stage.

**Parameters:**

| Name | Type | Description   |
| ---- | ---- | ------------- |
| None | —    | No parameters |

**Returns:**

| Type             | Description                                        |
| ---------------- | -------------------------------------------------- |
| pandas.DataFrame | Profiling breakdown including total execution time |

**Example Usage:**

```python
last_query_profile()
```

**Notes:**

* Requires `SET PROFILING = 1` to be enabled
* Adds a "Total Time" row for convenience

---

## 🔹 Function: `preview(table_name)`

**Description:**
Fetches the first 20 rows of a given table for quick inspection.

**Parameters:**

| Name       | Type | Description                  |
| ---------- | ---- | ---------------------------- |
| table_name | str  | Name of the table to preview |

**Returns:**

| Type             | Description                |
| ---------------- | -------------------------- |
| pandas.DataFrame | First 20 rows of the table |

**Example Usage:**

```python
preview("users")
```

**Notes:**

* Useful for quick data exploration
* Limit is fixed at 20 rows

---

💡 *These helper functions simplify SQL analysis, debugging, and data exploration within the notebook.*


-----

In [1]:
import pandas as pd
import sqlalchemy as sal

In [2]:
DB_USER = "shivam"
DB_PASSWORD = "your_password"
DB_HOST = "localhost"
DB_PORT = "3306"
DB_NAME = "ecommerce_db"

engine = sal.create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}"
)

In [3]:
def show(query):
    with engine.connect() as conn:
        result = conn.execute(sal.text(query))
        result_df = pd.DataFrame(result)
    return result_df

In [4]:
query = """
    USE ecommerce_db;
"""

In [5]:
with engine.connect() as conn:
        result = conn.execute(sal.text(query))

In [6]:
query = """
    SHOW tables;
"""

In [7]:
with engine.connect() as conn:
        tables = conn.execute(sal.text(query))
        tables = [table[0] for table in tables]

In [8]:
tables

['order_items', 'orders', 'payments', 'products', 'users']

In [9]:
table_schema = {}

In [10]:
for table in tables:
    query = f"""
        DESCRIBE
            {table}
    """
    with engine.connect() as conn:
        result = conn.execute(sal.text(query))
        result = pd.DataFrame(result)
        table_schema[table] = result

In [11]:
table_schema["users"]

,Field,Type,Null,Key,Default,Extra
0,user_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,email,text,YES,,None,
3,phone,text,YES,,None,
4,city,text,YES,,None,
5,country,text,YES,,None,
6,created_at,datetime,YES,,None,


In [12]:
orders = table_schema["orders"]
users = table_schema["users"]
products = table_schema["products"]
order_items = table_schema["order_items"]
payments = table_schema["payments"]

In [13]:
def explain(query):
    query = f"EXPLAIN ANALYZE {query};"
    with engine.connect() as conn:
        result = conn.execute(sal.text(query))
        print(f"\n{'='*20} QUERY PLAN {'='*20}\n")
        # result is an iterator of rows; the plan is usually in the first column
        for row in result:
            print(row[0])

------

In [14]:
query = """
    SET PROFILING = 1;
"""

In [15]:
with engine.connect() as conn:
        result = conn.execute(sal.text(query))

In [16]:
show("show profiles ;")

,Query_ID,Duration,Query
0,1,0.00017,ROLLBACK


In [17]:
def last_query_profile():
    with engine.connect() as conn:
        result = conn.execute(sal.text("SHOW PROFILES;"))
        result_df = pd.DataFrame(result)
        last_query_id = int(result_df["Query_ID"].max())
        
        new_query = f"""
            SHOW PROFILE FOR QUERY {last_query_id} ;
        """
        new_result = conn.execute(sal.text(new_query))
        new_query_df = pd.DataFrame(new_result)
        total_series = pd.Series({
            "Status": "Total Time", 
            "Duration": new_query_df["Duration"].sum()
        })
        new_query_df.loc[len(new_query_df)] = total_series
        return new_query_df

In [18]:
last_query_profile()

,Status,Duration
0,starting,0.000077
1,Executing hook on transaction,0.000005
2,starting,0.000083
3,query end,0.000016
4,closing tables,0.000005
5,freeing items,0.000058
6,cleaning up,0.000009
7,Total Time,0.000253


In [19]:
show("""
SELECT 
    TABLE_NAME,
    INDEX_NAME,
    COLUMN_NAME,
    NON_UNIQUE
FROM information_schema.STATISTICS
WHERE TABLE_SCHEMA = 'ecommerce_db'
ORDER BY TABLE_NAME, INDEX_NAME;
""")

,TABLE_NAME,INDEX_NAME,COLUMN_NAME,NON_UNIQUE
0,order_items,idx_order_items_order_id,order_id,1
1,order_items,idx_product_quantity,product_id,1
2,order_items,idx_product_quantity,quantity,1
3,order_items,PRIMARY,order_item_id,0
4,orders,idx_orders_date,order_date,1
5,orders,idx_orders_user_id,user_id,1
6,orders,PRIMARY,order_id,0
7,payments,idx_payments_order_id,order_id,1
8,payments,PRIMARY,payment_id,0
9,products,idx_products_price,price,1


In [20]:
with engine.connect() as conn:
    conn.execute(sal.text("USE ecommerce_db;"))

In [21]:
def preview(table_name):
    query = f"""
            SELECT
                *
            FROM
                {table_name}
            LIMIT 20;
    """
    with engine.connect() as conn:
        result = conn.execute(sal.text(query))
        return pd.DataFrame(result)    

----------------------------------------------------------------------------------------------------------------SETUP COMPLETE----------------------------------------------------------------------------------------------------------------

> Note: Used LIMIT 20 for faster query performance

## __Basic__

1. Select all columns from `users`.

In [21]:
query_1 = """
    SELECT
        *
    FROM
        users;
"""

In [22]:
show(query=query_1)

,user_id,name,email,phone,city,country,created_at
0,0000d817-629f-436b-b1e7-b71fe1102499,Misty Jackson,maldonadojacob@example.org,001-824-487-1933x09066,Victoriastad,Qatar,2023-09-18 10:21:35
1,000196c5-99a4-4c4a-96ef-3ba48a564b85,Michael Harris,georgeburton@example.net,(856)557-5675,Reynoldsburgh,Estonia,2020-03-26 13:21:42
2,00019ae4-fbf6-4589-840d-c142cbdcf552,David Durham,leejames@example.net,608.528.1438x097,Kingville,Antigua and Barbuda,2020-02-09 06:10:11
3,00021d12-efcb-41a4-86b0-dd4b08f359d8,Nicole Morales,hardyalex@example.com,001-325-376-7108,Levybury,Italy,2024-08-27 04:10:38
4,0003fccd-3076-41ef-beb4-28ecebcf9971,Jordan James,katrinalang@example.org,001-386-402-4096,East Karenbury,Saudi Arabia,2023-12-04 11:01:53
...,...,...,...,...,...,...,...
99995,fffb5578-20bf-4116-a4e4-944aa3b77051,Sarah Bryan,melanie79@example.com,(849)505-4957x4272,Port Carl,Pitcairn Islands,2021-02-21 02:03:43
99996,fffd011a-247c-43af-b483-cb452e8ed152,Lauren Williams DDS,paul26@example.org,(374)690-9269,Lake Shari,Tajikistan,2022-01-16 16:23:03
99997,fffd8db4-99f8-4ac2-a1f7-1b6fc905cd75,Michelle Bowen,kkim@example.org,+1-532-641-0759x6365,Duranstad,Argentina,2021-06-28 23:33:32
99998,fffed7eb-c868-4014-96cb-a5e7b5a9f3b6,Danielle West,zaustin@example.org,001-773-230-1469x27153,Webbmouth,Iran,2026-01-20 14:52:31


----

2. Select only `name` and `email` from users.

In [ ]:
query_2 = """
    SELECT
        name,
        email
    FROM
        users ;
"""

In [ ]:
show(query=query_2)

-----

3. Select all products.

In [ ]:
query_3 = """
    SELECT
        *
    FROM
        products;
"""

In [ ]:
show(query_3)

---

4. Display product names only.

In [ ]:
table_schema["products"]

In [ ]:
query_4 = """
    SELECT
        name
    FROM
        products;
"""

In [ ]:
show(query_4)

-----

5. Select `user_id` and `city` from users.

In [ ]:
query_5 = """
    SELECT
        user_id,
        city
    FROM
        users;
"""

In [ ]:
show(query_5)

------

6. Show all orders.

In [ ]:
tables

In [ ]:
query_6 = """
    SELECT
        *
    FROM
        orders;
"""

In [ ]:
show(query_6)

------

7. Display `order_id` and `status`.

In [ ]:
table_schema["orders"]

In [ ]:
query_7 = """
    SELECT
        order_id,
        status
    FROM
        orders;
"""

In [ ]:
show(query_7)

---

8. Select all payments.

In [ ]:
query_8 = """
    SELECT
        *
    FROM
        payments;
"""

In [ ]:
show(query_8)

----

9. Show product `name` and `price`.

In [ ]:
table_schema["products"]

In [ ]:
query_9 = """
    SELECT
        name,
        price
    FROM
        products;
"""

In [ ]:
show(query_9)

In [ ]:
explain(query_9)

---

10. Select all columns from `order_items`.

In [ ]:
query_10 = """
    SELECT
        *
    FROM
        order_items;
"""

In [ ]:
show(query_10)

---

### **2. WHERE**

11. Find users from 'India'.

In [ ]:
table_schema["users"]

In [ ]:
query_11 = """
    SELECT
        *
    FROM
        users u
    WHERE
        u.country = "India" ;
"""

In [ ]:
show(query_11)

In [ ]:
last_query_profile()

---

12. Get products with price > 1000.

In [ ]:
query_12 = """
    SELECT
        *
    FROM
        products p
    WHERE
        p.price > 1000 ;
"""

In [ ]:
show(query_12)

----

13. Find orders with status 'shipped'.

In [ ]:
query_13 = """
    SELECT
        *
    FROM
        orders o
    WHERE
        o.status = "shipped" ;
""" 

In [ ]:
show(query_13)

In [ ]:
last_query_profile()

---

14. Get payments with status 'failed'.

In [ ]:
tables

In [ ]:
table_schema["orders"]

In [ ]:
explain(query_13)

In [ ]:
query_14 = """
    SELECT
        *
    FROM
        payments;
"""

In [ ]:
show(query_14)

In [ ]:
last_query_profile()

----

15. Find users from 'USA' or 'Canada'.

In [19]:
table_schema["users"]

,Field,Type,Null,Key,Default,Extra
0,user_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,email,text,YES,,None,
3,phone,text,YES,,None,
4,city,text,YES,,None,
5,country,text,YES,,None,
6,created_at,datetime,YES,,None,


In [19]:
query_15 = """
    SELECT
        *
    FROM
        users
    WHERE
        country IN ("USA", "Canada") ;
"""

In [21]:
show(query_15)

,user_id,name,email,phone,city,country,created_at
0,01369757-86a6-4ad9-ba0c-f6f6d87a6288,Brandon Chen,kray@example.net,769-572-6276x46608,South Marktown,Canada,2021-02-06 17:49:32
1,0290bc6f-3eda-400f-aa55-060ab3f3b5a8,Lisa Jenkins,oboyd@example.net,252-515-4418,North Jaytown,Canada,2025-08-06 16:02:54
2,02e62e30-248a-439d-9905-691c05c1b0d4,Jared Nelson,molly61@example.org,001-318-341-0859x566,Fergusonberg,Canada,2025-02-01 08:22:23
3,0449c49d-7129-4333-8499-15e3e1271403,Jennifer Fischer,beverett@example.com,206-690-8630,Michealshire,Canada,2025-03-03 06:14:33
4,047bf34b-b59e-40cc-8e70-cc7ec95417b6,Kevin Bryant,alan19@example.net,273.993.5530x0656,South Lisa,Canada,2021-03-15 23:10:23
...,...,...,...,...,...,...,...
399,fb639165-90a8-438b-a735-f79245290263,William Espinoza,troy28@example.org,(335)544-7109x902,East Timothystad,Canada,2023-01-28 19:34:44
400,fb8cdaf5-8977-462d-8c70-ccf260e15e60,Rhonda Price,patrick67@example.com,(487)940-8161x8606,Yatesberg,Canada,2023-12-03 10:10:44
401,fca9bf53-0055-4783-8591-3eebf414175d,Sarah Flores,amanda85@example.org,(515)715-2717,New Sarah,Canada,2024-07-15 08:46:18
402,fd252ccd-7c8f-41c7-97e7-9b9bf7af201e,Susan Mckinney,ebeard@example.org,001-753-836-2470x72407,West Jeanne,Canada,2020-01-30 04:09:48


In [22]:
explain(query_15)

                                                                                                                                                                                                                   EXPLAIN
0  -> Filter: (users.country in ('USA','Canada'))  (cost=10821 rows=20888) (actual time=0.62..106 rows=404 loops=1)\n    -> Table scan on users  (cost=10821 rows=104441) (actual time=0.0385..89.5 rows=100000 loops=1)\n


----

16. Get products with stock < 50.

In [24]:
table_schema["products"]

,Field,Type,Null,Key,Default,Extra
0,product_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,category,text,YES,,None,
3,price,float,YES,,None,
4,stock,int,YES,,None,


In [25]:
query_16 = """
    SELECT
        *
    FROM
        products
    WHERE
        stock < 50 ;
"""

In [26]:
show(query_16)

,product_id,name,category,price,stock
0,002a6aee-e362-432a-bb34-7f4c266a3150,Debate,Clothing,176.57,11
1,0032bb3f-594a-422e-8e66-d33dcf28d764,Yet,Home,4978.13,33
2,007ca365-fb4d-4f46-bf03-8630174562c5,Inside,Clothing,1625.10,23
3,00a3e6bb-e209-4ca4-b7f2-c4fcaf6369be,Whose,Books,852.50,12
4,014935fb-4b82-457c-941f-c219eb56c71c,Experience,Electronics,2251.19,23
...,...,...,...,...,...
1022,fe40d57e-2fbd-4fc4-a9c3-e93d1a361e46,Spend,Home,79.43,10
1023,fe5dfc8f-17c6-4b3c-a24d-c612cdfcbeac,Son,Books,322.87,35
1024,fe5ef265-c5cf-4225-9662-3b96b36b2a3d,Value,Home,2929.60,46
1025,fe63a1a5-cb5b-4151-9904-3cfc17f21e3a,Executive,Sports,3042.57,41


----

17. Find orders placed after a certain date.

In [27]:
table_schema["orders"]

,Field,Type,Null,Key,Default,Extra
0,order_id,varchar(36),NO,PRI,None,
1,user_id,varchar(36),YES,MUL,None,
2,order_date,datetime,YES,,None,
3,status,text,YES,,None,


In [32]:
query_17 = """
    SELECT
        order_id,
        user_id,
        order_date,
        status
    FROM
        orders o 
    WHERE
        order_date > "2026-03-10";
"""

In [33]:
show(query_17)

,order_id,user_id,order_date,status
0,000015f8-ff52-41a7-bd5a-0f9eb1cd082e,40e6f1c0-2990-4256-8daa-4ac4a695ea1b,2026-03-24 20:53:23,placed
1,0000f575-ff08-4bd2-8006-c4aa34f569c3,8e821a83-9d41-4b02-947b-0fe753eb7567,2026-03-10 21:21:26,shipped
2,0001d9dc-ecf6-454c-b0a8-b0a1358d737d,7696b3f6-195a-4929-83f2-1d40c8bc91db,2026-03-24 02:06:41,cancelled
3,00028802-8fad-44c3-b074-db650331fab3,cc10f19a-10b4-4a92-8e5d-f8fc680c0e65,2026-03-25 03:32:24,delivered
4,00029ec0-6cf2-4b07-9a1a-d9d37400269b,9c1c15ce-6e05-4860-a836-c908ee152129,2026-03-14 19:59:46,shipped
...,...,...,...,...
58716,fff840f4-4ee6-48dc-8837-12cab0db1cbf,406825ef-ded7-4be5-a01c-5e832fe8bf7d,2026-03-13 23:55:18,delivered
58717,fff9d85a-6d5b-4df1-99f7-cdebaed909c2,27eeeb81-0853-4521-9a3a-283080606a12,2026-03-26 08:42:28,shipped
58718,fffb50ba-bc96-4d02-88e8-e5730769e1db,64ac560a-4a86-47a5-bb83-d65c16950de3,2026-03-21 16:55:27,placed
58719,fffd2359-c5fe-470e-b102-fefa5ef064af,9a064795-52dd-487a-b58a-0d95b4ca6053,2026-03-17 07:19:10,shipped


---

18. Get payments above 1000.

In [35]:
table_schema["payments"]

,Field,Type,Null,Key,Default,Extra
0,payment_id,varchar(36),NO,PRI,None,
1,order_id,varchar(36),YES,MUL,None,
2,amount,float,YES,,None,
3,payment_method,text,YES,,None,
4,payment_status,text,YES,,None,
5,payment_date,datetime,YES,,None,


In [36]:
query_18 = """
    SELECT
        *
    FROM
        payments p
    WHERE
        amount > 1000;
"""

In [37]:
show(query_18)

,payment_id,order_id,amount,payment_method,payment_status,payment_date
0,0000803a-bfb4-4dfa-81fe-10884fd5ae37,6799a061-d1bb-41da-8e80-8eeb76fe7f5f,1806.55,wallet,success,2026-01-10 01:23:35
1,0000ce28-6059-41fe-807d-5dc6c602779f,d71d0ed5-4ee7-48cc-935b-95b8d83c2943,4987.84,card,pending,2026-01-01 06:42:21
2,0000da01-dadb-46d1-9f7b-40c627f8fc09,d34e6141-6770-491a-973a-3a686b94134d,3330.91,wallet,failed,2026-03-22 11:23:36
3,00012b70-cdee-4584-8b06-e324f07b21c2,c555a2f6-c024-4774-9db7-5f92ad00029a,4913.13,wallet,pending,2026-02-07 11:47:42
4,00014e00-2c00-444f-8b5b-4e57927ce4b2,776dc260-2c72-41b9-8c46-8944fc680a68,1305.83,netbanking,success,2026-03-09 14:57:08
...,...,...,...,...,...,...
240673,fffec9d4-d93a-4315-83f1-327917f562d6,0cf8af09-7418-4310-a4be-3bf404158bc6,2885.89,netbanking,pending,2026-01-07 10:52:31
240674,ffff30eb-c19d-49ca-8f40-57aa70109ab6,55ff8448-9e22-4c2b-8c14-441cb15f28f4,3989.09,upi,pending,2026-01-07 20:17:42
240675,ffff8ee2-e6d4-4c5c-8d91-d8109adb4dff,1945e51d-ed9a-46ab-b9d2-918ac57685d2,2148.18,wallet,failed,2026-01-09 01:07:31
240676,ffff9153-406d-4a09-983e-2e7d14a11d2d,f049d2b5-d58b-42a9-9a52-5c94922fabd5,3836.75,card,pending,2026-02-05 18:51:47


---

19. Find users not from 'India'

In [42]:
table_schema["users"]

,Field,Type,Null,Key,Default,Extra
0,user_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,email,text,YES,,None,
3,phone,text,YES,,None,
4,city,text,YES,,None,
5,country,text,YES,,None,
6,created_at,datetime,YES,,None,


In [43]:
query_19 = """
    SELECT
        *
    FROM
        users u
    WHERE
        u.country NOT LIKE "India" ;
"""

In [44]:
show(query_19)

,user_id,name,email,phone,city,country,created_at
0,0000d817-629f-436b-b1e7-b71fe1102499,Misty Jackson,maldonadojacob@example.org,001-824-487-1933x09066,Victoriastad,Qatar,2023-09-18 10:21:35
1,000196c5-99a4-4c4a-96ef-3ba48a564b85,Michael Harris,georgeburton@example.net,(856)557-5675,Reynoldsburgh,Estonia,2020-03-26 13:21:42
2,00019ae4-fbf6-4589-840d-c142cbdcf552,David Durham,leejames@example.net,608.528.1438x097,Kingville,Antigua and Barbuda,2020-02-09 06:10:11
3,00021d12-efcb-41a4-86b0-dd4b08f359d8,Nicole Morales,hardyalex@example.com,001-325-376-7108,Levybury,Italy,2024-08-27 04:10:38
4,0003fccd-3076-41ef-beb4-28ecebcf9971,Jordan James,katrinalang@example.org,001-386-402-4096,East Karenbury,Saudi Arabia,2023-12-04 11:01:53
...,...,...,...,...,...,...,...
99620,fffb5578-20bf-4116-a4e4-944aa3b77051,Sarah Bryan,melanie79@example.com,(849)505-4957x4272,Port Carl,Pitcairn Islands,2021-02-21 02:03:43
99621,fffd011a-247c-43af-b483-cb452e8ed152,Lauren Williams DDS,paul26@example.org,(374)690-9269,Lake Shari,Tajikistan,2022-01-16 16:23:03
99622,fffd8db4-99f8-4ac2-a1f7-1b6fc905cd75,Michelle Bowen,kkim@example.org,+1-532-641-0759x6365,Duranstad,Argentina,2021-06-28 23:33:32
99623,fffed7eb-c868-4014-96cb-a5e7b5a9f3b6,Danielle West,zaustin@example.org,001-773-230-1469x27153,Webbmouth,Iran,2026-01-20 14:52:31


---

20. Get products with price between 100 and 500.

In [48]:
query_20 = """
    SELECT
        name
    FROM
        products p
    WHERE
        p.price BETWEEN 100 AND 500;
"""

In [50]:
show(query_20)

,name
0,Western
1,Debate
2,Score
3,Difference
4,Him
...,...
1597,Other
1598,While
1599,Manager
1600,Into


-----

## __3. ORDER BY & LIMIT__

21. Sort products by price ascending.

In [51]:
query_21 = """
    SELECT
        *
    FROM
        products p
    ORDER BY
        p.price ;
"""

In [52]:
show(query_21)

,product_id,name,category,price,stock
0,266326e5-6492-4bc3-988c-f7c729f68211,Vote,Clothing,5.01,699
1,0809edd4-3e03-4b89-97c4-2aa9d703b17a,Reason,Sports,5.32,468
2,5db5839c-5e87-497f-a43a-62df5e5e530a,Total,Home,5.96,782
3,a282f6db-2e57-405e-8dea-a9e30116c341,Agreement,Sports,6.55,489
4,ead84b6a-dc41-4d73-af9e-dd1395941b12,Anything,Books,6.81,655
...,...,...,...,...,...
19995,4a199d88-d324-4d5d-a275-2ec2ce0e2655,Minute,Sports,4999.43,170
19996,1be71191-46b0-4a04-8b72-0841a922fc25,Less,Books,4999.53,596
19997,db40bc48-328f-4eb5-8a65-bb0dec061dfb,Result,Electronics,4999.71,798
19998,882fab09-1abc-49e0-a7db-0f1681f2dbb1,My,Sports,4999.97,417


---

22. Sort products by price descending.

In [53]:
query_22 = """
    SELECT
        *
    FROM
        products p
    ORDER BY
        p.price DESC;
"""

In [54]:
show(query_22)

,product_id,name,category,price,stock
0,c5edb425-bda5-4f14-86e4-29b3f609fc10,Civil,Clothing,4999.98,57
1,882fab09-1abc-49e0-a7db-0f1681f2dbb1,My,Sports,4999.97,417
2,db40bc48-328f-4eb5-8a65-bb0dec061dfb,Result,Electronics,4999.71,798
3,1be71191-46b0-4a04-8b72-0841a922fc25,Less,Books,4999.53,596
4,4a199d88-d324-4d5d-a275-2ec2ce0e2655,Minute,Sports,4999.43,170
...,...,...,...,...,...
19995,ead84b6a-dc41-4d73-af9e-dd1395941b12,Anything,Books,6.81,655
19996,a282f6db-2e57-405e-8dea-a9e30116c341,Agreement,Sports,6.55,489
19997,5db5839c-5e87-497f-a43a-62df5e5e530a,Total,Home,5.96,782
19998,0809edd4-3e03-4b89-97c4-2aa9d703b17a,Reason,Sports,5.32,468


---

23. Get top 5 cheapest products.

In [55]:
table_schema["products"]

,Field,Type,Null,Key,Default,Extra
0,product_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,category,text,YES,,None,
3,price,float,YES,,None,
4,stock,int,YES,,None,


In [56]:
query_23 = """
    SELECT
        *
    FROM
        products p
    ORDER BY
        p.price
    LIMIT 5;
"""

In [57]:
show(query_23)

,product_id,name,category,price,stock
0,266326e5-6492-4bc3-988c-f7c729f68211,Vote,Clothing,5.01,699
1,0809edd4-3e03-4b89-97c4-2aa9d703b17a,Reason,Sports,5.32,468
2,5db5839c-5e87-497f-a43a-62df5e5e530a,Total,Home,5.96,782
3,a282f6db-2e57-405e-8dea-a9e30116c341,Agreement,Sports,6.55,489
4,ead84b6a-dc41-4d73-af9e-dd1395941b12,Anything,Books,6.81,655


-----

24. Get top 10 most expensive products.

In [58]:
query_24 = """
    SELECT
        *
    FROM
        products p
    ORDER BY
        p.price DESC
    LIMIT 10;
"""

In [59]:
show(query_24)

,product_id,name,category,price,stock
0,c5edb425-bda5-4f14-86e4-29b3f609fc10,Civil,Clothing,4999.98,57
1,882fab09-1abc-49e0-a7db-0f1681f2dbb1,My,Sports,4999.97,417
2,db40bc48-328f-4eb5-8a65-bb0dec061dfb,Result,Electronics,4999.71,798
3,1be71191-46b0-4a04-8b72-0841a922fc25,Less,Books,4999.53,596
4,4a199d88-d324-4d5d-a275-2ec2ce0e2655,Minute,Sports,4999.43,170
5,8befcd10-4525-46e2-aafa-ca71936855a2,Provide,Electronics,4999.31,173
6,0f4b500f-d323-407a-a540-b913e6dac977,Bed,Books,4999.20,680
7,f61fe562-9026-4c84-826e-7862bb4fed9e,Real,Books,4999.07,934
8,196df4f2-e6c6-4837-8b2e-ee510b0a5d45,Finally,Sports,4998.17,177
9,f8d0615e-6b2c-4b91-a3db-8cf58b37732d,Pm,Electronics,4997.95,98


---

25. Sort users by name.

In [60]:
table_schema["users"]

,Field,Type,Null,Key,Default,Extra
0,user_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,email,text,YES,,None,
3,phone,text,YES,,None,
4,city,text,YES,,None,
5,country,text,YES,,None,
6,created_at,datetime,YES,,None,


In [61]:
query_25 = """
    SELECT
        *
    FROM
        users u
    ORDER BY
        name;
"""

In [62]:
show(query_25)

,user_id,name,email,phone,city,country,created_at
0,ab99bbd6-1528-4ae9-8cc4-96b1a21a489f,Aaron Aguilar,richardjames@example.com,(569)228-4394,Ramseyfort,Slovakia (Slovak Republic),2023-08-03 11:50:47
1,78375196-bf2d-42ba-ac71-ca55f12580a6,Aaron Aguilar,bbennett@example.org,+1-592-754-1777x984,Dudleyfort,Equatorial Guinea,2021-04-15 01:36:31
2,67e74a3e-b9c4-494b-a830-ea63a15e87e7,Aaron Ali,kevin88@example.org,(250)937-0260,Sanchezshire,Slovenia,2020-07-09 00:42:22
3,9b3df6a2-52db-491a-b979-a3b6060bf456,Aaron Allen,christopherchapman@example.org,674-350-2742x59933,Johnsonmouth,Maldives,2023-05-23 05:54:17
4,776c35dc-d838-4863-b295-04cf9c1ea5de,Aaron Alvarado,alyssa72@example.net,+1-249-241-2572x292,Port Megan,Tonga,2021-01-13 21:23:26
...,...,...,...,...,...,...,...
99995,c1f9a1f2-d90c-4bb6-ad90-68c098a37433,Zoe Rivera,jeffery73@example.org,001-409-510-2197x9237,South Richardview,Ireland,2024-11-25 14:00:54
99996,e2bdd287-97eb-4887-a6c3-cdd4940c44df,Zoe Roberts,brent95@example.net,7064944980,Bonniehaven,China,2021-05-24 16:26:24
99997,f0fc6314-f7f7-4871-b60c-1e4a0d2e7596,Zoe Scott,sarah63@example.org,446-717-6944x2714,Michaeltown,Mozambique,2021-07-23 12:49:04
99998,3913364e-3f98-4180-afe6-f08131c9a660,Zoe Singh,shawdiana@example.org,001-432-411-1769,East Mary,Botswana,2020-06-26 06:51:05


---

26. Show latest 10 orders.

In [63]:
query_26 = """
    SELECT
        *
    FROM
        orders o
    ORDER BY
        order_date DESC
    LIMIT 10;
"""

In [64]:
show(query_26)

,order_id,user_id,order_date,status
0,9da99009-06e3-4c93-bd4e-05cb1e95f336,5999fd76-1bad-4903-8908-047ff818afeb,2026-03-26 13:37:00,placed
1,df07e248-b937-4694-9fd3-9e94c21cc003,fcebc43f-a013-4b86-b6ca-d8a747f2de79,2026-03-26 13:36:42,delivered
2,e40db809-c8fe-4916-94cf-604dcba7dfa6,3f01bc1c-f001-4b28-8493-9b0dd76f7e12,2026-03-26 13:36:34,delivered
3,cddc6186-33c1-4199-99db-d83e9cc1d101,f98327d6-62aa-4bc3-ace4-7fc2bfa99d6d,2026-03-26 13:36:18,shipped
4,7a7e60f6-816a-499f-b96e-fc9870f1b1da,2be0c7e2-cd11-4db2-8b3f-76ebc752c32c,2026-03-26 13:36:08,cancelled
5,0b950600-6a6d-4d62-9116-572b18cfed4a,08bee92e-df80-4d19-8937-2e917d43d9f3,2026-03-26 13:35:45,placed
6,e3148021-8a27-4807-9528-e1fb7ae5ba33,4c940a17-f741-435e-8264-3f0ec97fb03d,2026-03-26 13:35:12,shipped
7,a2cca3d3-c3ff-4989-87ed-baa242a56418,597633db-d91e-4dba-a806-7200df630505,2026-03-26 13:34:56,cancelled
8,75f79595-ef71-4240-b847-e73ddfeb1839,307adaf2-117f-44aa-aa0f-5a245362060f,2026-03-26 13:34:53,placed
9,d42ace4a-269a-4347-8a93-b79f80c1154c,4d9dc7b3-0ada-4fae-a328-2b95a418abe7,2026-03-26 13:34:23,placed


---

27. Sort payments by amount.

In [65]:
query_27 = """
    SELECT
        *
    FROM
        payments p
    ORDER BY
        p.amount ;
"""

In [66]:
show(query_27)

,payment_id,order_id,amount,payment_method,payment_status,payment_date
0,aaca186b-cbca-4b66-b9bc-61e50c07616f,6b77c0fe-7046-4aec-99d7-c47e0ed6e380,10.01,card,success,2026-01-24 16:44:21
1,3aee5f31-57c5-4ad6-836d-3a1799320003,13cee1e9-596d-4d33-89c4-773a93c30c90,10.01,card,pending,2026-01-31 05:14:47
2,95f66c29-dc7a-4588-9dfb-684cfb4d9299,9b0631f6-f899-43ea-858f-b661883c94ec,10.02,upi,success,2026-01-08 10:33:58
3,fc201a32-fc02-4c78-b104-d2f7c2895236,11b81c0b-a33d-42fa-9307-bc194eee2351,10.02,netbanking,pending,2026-03-03 04:19:51
4,ef0b2aec-0a2e-4686-9c06-543e7daff383,7d627bd9-8a75-4d36-a5fe-fb5754cae2b8,10.12,upi,failed,2026-01-31 08:07:56
...,...,...,...,...,...,...
299995,7010c5c5-e9d4-4aea-8961-f1480246cc66,94b7e09d-ea2f-438b-831f-36faf200c960,4999.90,upi,success,2026-02-22 00:24:38
299996,ce60e78b-e67a-4fd8-9d0a-fbf9e499a1c8,e679e537-57b0-4c98-895f-23defd536938,4999.95,netbanking,pending,2026-01-22 15:28:46
299997,c89d83fa-1bd3-4c87-9d25-9976221a3a47,cc64fbb1-be8a-4c67-bc90-a0f8c50d3b5f,4999.95,upi,pending,2026-02-16 08:10:17
299998,9a664e56-1e73-46e1-b05f-297098f6946a,104c7bb9-435b-4846-aefc-39d70e98535d,4999.96,wallet,failed,2026-03-09 05:24:39


---

28. Get top 3 highest payments.

In [69]:
query_28 = """
    SELECT
        *
    FROM
        payments p
    ORDER BY
        p.amount DESC
    LIMIT
        3;
"""

In [70]:
show(query_28)

,payment_id,order_id,amount,payment_method,payment_status,payment_date
0,3d1f10a0-c7ac-478c-9318-21f45e64be70,48510c98-bced-4aed-9f30-a4113f2b9d68,5000.00,upi,success,2026-02-24 03:08:10
1,9a664e56-1e73-46e1-b05f-297098f6946a,104c7bb9-435b-4846-aefc-39d70e98535d,4999.96,wallet,failed,2026-03-09 05:24:39
2,ce60e78b-e67a-4fd8-9d0a-fbf9e499a1c8,e679e537-57b0-4c98-895f-23defd536938,4999.95,netbanking,pending,2026-01-22 15:28:46


----

29. Sort users by created_at.

In [71]:
query_29 = """
    SELECT
        *
    FROM
        users u
    ORDER BY
        u.created_at;
"""

In [72]:
show(query_29)

,user_id,name,email,phone,city,country,created_at
0,4fe4787e-e1c9-472f-9746-ef7176162573,Chad Gonzalez,megan27@example.org,397-886-4639x91501,Garretttown,Central African Republic,2020-01-01 01:13:38
1,bbd1c67c-c6aa-431e-b915-f6bdffdb7c53,Samuel Bradshaw,vbrown@example.com,001-795-376-6447x755,Tracyfort,Samoa,2020-01-01 03:34:34
2,af6e1502-ea9d-4d0d-9110-b957af192625,Travis Doyle,robertdaniels@example.org,481-573-0373x8410,New David,Azerbaijan,2020-01-01 03:38:12
3,1f407ee3-f751-4028-bf37-70e3eae5cc01,Kimberly Lewis,haysmelissa@example.net,001-922-570-0714,Samanthaton,Kazakhstan,2020-01-01 03:54:08
4,ae4b7606-13ae-422d-a354-9c9014430fed,James Cameron,thomasjohn@example.com,873.431.5539x356,Port Jasmine,Burkina Faso,2020-01-01 04:52:12
...,...,...,...,...,...,...,...
99995,28e71d75-c16c-4f1b-8bfa-f31ed6fe48d1,Timothy Murray,xmartin@example.com,6169986133,Williamschester,Martinique,2026-03-26 09:31:00
99996,d096dae5-614e-4ecd-8927-bceafcc02dc5,Theresa Welch,allen82@example.com,4855162297,Lake Jose,Turkey,2026-03-26 10:04:37
99997,a14182d6-7762-47aa-a03a-b6cee78cd024,Rebecca Ramirez,davenportjessica@example.org,(545)384-1214x709,East Andrew,Saudi Arabia,2026-03-26 10:56:39
99998,37f42cda-0c5c-43de-b35d-823d97f7d51e,Jessica Fischer,jessica25@example.net,956.508.6637x94433,Morganfort,Zambia,2026-03-26 11:49:45


----

30. Show first 20 products.

In [73]:
table_schema["products"]

,Field,Type,Null,Key,Default,Extra
0,product_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,category,text,YES,,None,
3,price,float,YES,,None,
4,stock,int,YES,,None,


In [74]:
query_30 = """
    SELECT
        *
    FROM
        products p
    LIMIT 20;
"""

In [75]:
show(query_30)

,product_id,name,category,price,stock
0,0003c4a5-212c-47ef-af0f-cdee99c413c5,Western,Electronics,352.58,819
1,0006cdee-589d-4643-bc5b-a4706b439d5a,Cup,Clothing,4713.70,872
2,00081441-ccf7-48a3-ba59-c3552457b66b,Mouth,Sports,74.22,83
3,00175b94-a20d-4273-8b80-edaf89168774,Stay,Electronics,684.80,955
4,001a274d-067e-4cc2-92f7-7521cb51ed43,Within,Books,849.07,892
5,0020fd29-307b-45ee-b625-ddfda55b91bd,Project,Electronics,3634.29,602
6,0022a06a-d276-490b-beb4-93e747b5b8c9,Sound,Sports,1447.02,851
7,00238654-2aea-47c4-afe8-6f1bcae16246,Decade,Home,3449.32,235
8,00245323-f4ee-44c0-b518-80dc50ba906f,Deal,Clothing,2672.84,310
9,00246097-550b-4c37-912d-a884d8ffa959,War,Sports,1613.24,690


-----

## __4. DISTINCT__

31. Get unique countries from users.

In [76]:
query_31 = """
    SELECT DISTINCT
        country
    FROM
        users u ;
"""

In [77]:
show(query_31)

,country
0,Qatar
1,Estonia
2,Antigua and Barbuda
3,Italy
4,Saudi Arabia
...,...
238,Saint Pierre and Miquelon
239,Greece
240,Czech Republic
241,Christmas Island


------

32. Get unique cities.

In [13]:
table_schema["users"]

,Field,Type,Null,Key,Default,Extra
0,user_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,email,text,YES,,None,
3,phone,text,YES,,None,
4,city,text,YES,,None,
5,country,text,YES,,None,
6,created_at,datetime,YES,,None,


In [14]:
query_32 = """
    SELECT DISTINCT
        city
    FROM
        users u;
"""

In [15]:
show(query_32)

,city
0,Victoriastad
1,Reynoldsburgh
2,Kingville
3,Levybury
4,East Karenbury
...,...
38166,North Michaelaville
38167,Lake Kathystad
38168,East Cheyenneshire
38169,Lake Shari


---

33. Get distinct product categories.

In [16]:
query_33 = """
    SELECT DISTINCT
        category
    FROM
        products p;
"""

In [17]:
show(query_33)

,category
0,Electronics
1,Clothing
2,Sports
3,Books
4,Home


----

34. Get distinct order statuses.

In [18]:
query_34 = """
    SELECT DISTINCT
        status
    FROM
        orders o ;
"""

In [19]:
show(query_34)

,status
0,placed
1,shipped
2,delivered
3,cancelled


----

35. Get unique payment methods.

In [20]:
table_schema["payments"]

,Field,Type,Null,Key,Default,Extra
0,payment_id,varchar(36),NO,PRI,None,
1,order_id,varchar(36),YES,MUL,None,
2,amount,float,YES,,None,
3,payment_method,text,YES,,None,
4,payment_status,text,YES,,None,
5,payment_date,datetime,YES,,None,


In [21]:
query_35 = """
    SELECT DISTINCT
        payment_method
    FROM
        payments p ;
"""

In [22]:
show(query_35)

,payment_method
0,wallet
1,upi
2,card
3,netbanking


----

36. Get distinct user_ids from orders.

In [25]:
table_schema["users"]

,Field,Type,Null,Key,Default,Extra
0,user_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,email,text,YES,,None,
3,phone,text,YES,,None,
4,city,text,YES,,None,
5,country,text,YES,,None,
6,created_at,datetime,YES,,None,


In [26]:
query_36 = """
    SELECT DISTINCT
        user_id
    FROM
        users;
"""

In [27]:
show(query_36)

,user_id
0,0000d817-629f-436b-b1e7-b71fe1102499
1,000196c5-99a4-4c4a-96ef-3ba48a564b85
2,00019ae4-fbf6-4589-840d-c142cbdcf552
3,00021d12-efcb-41a4-86b0-dd4b08f359d8
4,0003fccd-3076-41ef-beb4-28ecebcf9971
...,...
99995,fffb5578-20bf-4116-a4e4-944aa3b77051
99996,fffd011a-247c-43af-b483-cb452e8ed152
99997,fffd8db4-99f8-4ac2-a1f7-1b6fc905cd75
99998,fffed7eb-c868-4014-96cb-a5e7b5a9f3b6


---

37. Get distinct product_ids from order_items.

In [28]:
table_schema["products"]

,Field,Type,Null,Key,Default,Extra
0,product_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,category,text,YES,,None,
3,price,float,YES,,None,
4,stock,int,YES,,None,


In [29]:
query_37 = """
    SELECT DISTINCT
        product_id
    FROM
        products p ;
"""

In [30]:
show(query_37)

,product_id
0,0003c4a5-212c-47ef-af0f-cdee99c413c5
1,0006cdee-589d-4643-bc5b-a4706b439d5a
2,00081441-ccf7-48a3-ba59-c3552457b66b
3,00175b94-a20d-4273-8b80-edaf89168774
4,001a274d-067e-4cc2-92f7-7521cb51ed43
...,...
19995,ffed0951-9978-4eee-a208-d16feff97b15
19996,fff07745-7559-49e6-bef5-30dd510e3507
19997,fff80c67-088d-43b1-a46d-47d3a66542c9
19998,fff98964-41ec-4a37-b250-0bd56528040f


---

38. Get distinct payment statuses.

In [31]:
table_schema["payments"]

,Field,Type,Null,Key,Default,Extra
0,payment_id,varchar(36),NO,PRI,None,
1,order_id,varchar(36),YES,MUL,None,
2,amount,float,YES,,None,
3,payment_method,text,YES,,None,
4,payment_status,text,YES,,None,
5,payment_date,datetime,YES,,None,


In [33]:
query_38 = """
    SELECT DISTINCT
        payment_status
    FROM
        payments p ;
"""

In [34]:
show(query_38)

,payment_status
0,success
1,pending
2,failed


---

39. Get unique stock values.

In [35]:
table_schema["products"]

,Field,Type,Null,Key,Default,Extra
0,product_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,category,text,YES,,None,
3,price,float,YES,,None,
4,stock,int,YES,,None,


In [36]:
query_39 = """
    SELECT DISTINCT
        stock
    FROM
        products p;
"""

In [37]:
show(query_39)

,stock
0,819
1,872
2,83
3,955
4,892
...,...
996,540
997,742
998,418
999,103


---

40. Get distinct combinations of city and country.

In [38]:
query_40 = """
    SELECT DISTINCT
        city,
        country
    FROM
        users u ;
"""

In [39]:
show(query_40)

,city,country
0,Victoriastad,Qatar
1,Reynoldsburgh,Estonia
2,Kingville,Antigua and Barbuda
3,Levybury,Italy
4,East Karenbury,Saudi Arabia
...,...,...
98411,Port Carl,Pitcairn Islands
98412,Lake Shari,Tajikistan
98413,Duranstad,Argentina
98414,Webbmouth,Iran


---

## __5. AGGREGATIONS__


41. Count total users.

In [40]:
query_41 = """
    SELECT
        count(*)
    FROM
        users u;
"""

In [41]:
show(query_41)

,count(*)
0,100000


In [42]:
explain(query_41)

                                                         EXPLAIN
0  -> Count rows in u  (actual time=5.63..5.63 rows=1 loops=1)\n


---

42. Count total products.

In [43]:
table_schema["products"]

,Field,Type,Null,Key,Default,Extra
0,product_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,category,text,YES,,None,
3,price,float,YES,,None,
4,stock,int,YES,,None,


In [56]:
query_42 = """
    SELECT
        count(product_id) total_products
    FROM
        products p ;
"""

In [57]:
show(query_42)

,total_products
0,20000


---

43. Count total orders.

In [58]:
query_43 = """
    SELECT
        count(*) total_orders
    FROM
        orders o;
"""

In [59]:
show(query_43)

,total_orders
0,300000


---

44. Find average product price.

In [62]:
query_44 = """
    SELECT
        AVG(p.price) as avg_price
    FROM
        products p ;
"""

In [63]:
show(query_44)

,avg_price
0,2505.856876


---

45. Find max product price.

In [64]:
query_45 = """
    SELECT
        MAX(p.price) max_price
    FROM
        products p ;
"""

In [65]:
show(query_45)

,max_price
0,4999.98


---

46. Find min product price.

In [66]:
query_46 = """
    SELECT
        MIN(p.price) min_price
    FROM
        products p ;
"""

In [67]:
show(query_46)

,min_price
0,5.01


---

47. Sum all payment amounts.

In [68]:
table_schema["payments"]

,Field,Type,Null,Key,Default,Extra
0,payment_id,varchar(36),NO,PRI,None,
1,order_id,varchar(36),YES,MUL,None,
2,amount,float,YES,,None,
3,payment_method,text,YES,,None,
4,payment_status,text,YES,,None,
5,payment_date,datetime,YES,,None,


In [79]:
query_47 = """
    SELECT
        CAST(SUM(p.amount) AS DECIMAL) sum_amount
    FROM
        payments p ;
"""

In [80]:
show(query_47)

,sum_amount
0,751909613


---

48. Count failed payments.

In [83]:
table_schema["payments"]

,Field,Type,Null,Key,Default,Extra
0,payment_id,varchar(36),NO,PRI,None,
1,order_id,varchar(36),YES,MUL,None,
2,amount,float,YES,,None,
3,payment_method,text,YES,,None,
4,payment_status,text,YES,,None,
5,payment_date,datetime,YES,,None,


In [84]:
query_48 = """
    SELECT
        count(payment_id) failed_payments
    FROM
        payments p
    WHERE
        p.payment_status = "failed";
"""

In [85]:
show(query_48)

,failed_payments
0,99924


-----

49. Find average order amount.

In [91]:
table_schema["products"]

,Field,Type,Null,Key,Default,Extra
0,product_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,category,text,YES,,None,
3,price,float,YES,,None,
4,stock,int,YES,,None,


In [94]:
query_49 = """
    SELECT
        AVG(p.amount) average_amount
    FROM
        payments p;

"""

In [95]:
show(query_49)

,average_amount
0,2506.365376


---

50. Count total order_items.

In [98]:
query_50 = """
    SELECT
        count(*) total_order_items 
    FROM
        order_items;
"""

In [99]:
show(query_50)

,total_order_items
0,899816


----

## __6. GROUP BY__

51. Count users per country.

In [103]:
query_51 = """
    SELECT
        country,
        count(user_id) as no_of_users
    FROM
        users u
    GROUP BY
        u.country;
"""

In [104]:
show(query_51)

,country,no_of_users
0,Qatar,397
1,Estonia,376
2,Antigua and Barbuda,438
3,Italy,420
4,Saudi Arabia,375
...,...,...
238,Saint Pierre and Miquelon,419
239,Greece,400
240,Czech Republic,406
241,Christmas Island,395


---

52. Count products per category.

In [106]:
query_52 = """
    SELECT
        p.category,
        COUNT(p.product_id) no_of_products
    FROM
        products p
    GROUP BY
        p.category ;
"""

In [107]:
show(query_52)

,category,no_of_products
0,Electronics,4050
1,Clothing,3961
2,Sports,3932
3,Books,4062
4,Home,3995




----

53. Count orders per status.

In [108]:
query_53 = """
    SELECT
        o.status,
        COUNT(o.order_id) number_of_orders
    FROM
        orders o
    GROUP BY
        o.status ;
"""

In [109]:
show(query_53)

,status,number_of_orders
0,placed,74931
1,shipped,75278
2,delivered,74913
3,cancelled,74878


---

54. Count orders per user.

In [110]:
table_schema["users"]

,Field,Type,Null,Key,Default,Extra
0,user_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,email,text,YES,,None,
3,phone,text,YES,,None,
4,city,text,YES,,None,
5,country,text,YES,,None,
6,created_at,datetime,YES,,None,


In [111]:
table_schema["orders"]

,Field,Type,Null,Key,Default,Extra
0,order_id,varchar(36),NO,PRI,None,
1,user_id,varchar(36),YES,MUL,None,
2,order_date,datetime,YES,,None,
3,status,text,YES,,None,


In [112]:
query_54 = """
    SELECT
        user_id,
        COUNT(order_id) no_of_orders
    FROM
        orders o
    GROUP BY
        o.user_id;
"""

In [113]:
show(query_54)

,user_id,no_of_orders
0,0000d817-629f-436b-b1e7-b71fe1102499,1
1,000196c5-99a4-4c4a-96ef-3ba48a564b85,2
2,00019ae4-fbf6-4589-840d-c142cbdcf552,1
3,00021d12-efcb-41a4-86b0-dd4b08f359d8,1
4,0003fccd-3076-41ef-beb4-28ecebcf9971,1
...,...,...
95052,fffb5578-20bf-4116-a4e4-944aa3b77051,6
95053,fffd011a-247c-43af-b483-cb452e8ed152,1
95054,fffd8db4-99f8-4ac2-a1f7-1b6fc905cd75,4
95055,fffed7eb-c868-4014-96cb-a5e7b5a9f3b6,2


---

55. Sum quantity per product.

In [114]:
table_schema["products"]

,Field,Type,Null,Key,Default,Extra
0,product_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,category,text,YES,,None,
3,price,float,YES,,None,
4,stock,int,YES,,None,


In [115]:
query_55 = """
    SELECT 
        product_id, 
        SUM(quantity) 
    FROM 
        order_items
    GROUP BY 
        product_id;
"""

In [117]:
# show(query_55)

----

56. Sum payments per method.

In [118]:
query_56 = """
    SELECT
        p.payment_method,
        SUM(p.amount) as sum_of_amount
    FROM
        payments p
    GROUP BY
        p.payment_method ;
"""

In [120]:
show(query_56)

,payment_method,sum_of_amount
0,wallet,1.879608e+08
1,upi,1.877340e+08
2,card,1.890830e+08
3,netbanking,1.871318e+08


---

57. Count payments per status.

In [124]:
table_schema["payments"]

,Field,Type,Null,Key,Default,Extra
0,payment_id,varchar(36),NO,PRI,None,
1,order_id,varchar(36),YES,MUL,None,
2,amount,float,YES,,None,
3,payment_method,text,YES,,None,
4,payment_status,text,YES,,None,
5,payment_date,datetime,YES,,None,


In [127]:
query_57 = """
    SELECT
        p.payment_status,
        COUNT(p.payment_id) as no_of_payment
    FROM
        payments p
    GROUP BY
        p.payment_status ;
"""

In [128]:
show(query_57)

,payment_status,no_of_payment
0,success,100197
1,pending,99879
2,failed,99924


----

58. Avg price per category.

In [130]:
query_58 = """
    SELECT
        p.category,
        AVG(p.price)
    FROM
        products p
    GROUP BY
        p.category ;
"""

In [131]:
show(query_58)

,category,AVG(p.price)
0,Electronics,2502.413220
1,Clothing,2500.986786
2,Sports,2518.002610
3,Books,2493.748187
4,Home,2514.534146


---

59. Max price per category.

In [133]:
query_59 = """
    SELECT
        p.category,
        MAX(p.price)
    FROM
        products p
    GROUP BY
        p.category ;
"""

In [134]:
show(query_59)

,category,MAX(p.price)
0,Electronics,4999.71
1,Clothing,4999.98
2,Sports,4999.97
3,Books,4999.53
4,Home,4997.36


---

60. Count users per city.

In [135]:
query_60 = """
    SELECT
        city,
        COUNT(user_id)
    FROM
        users u
    GROUP BY
        u.city;
"""

In [136]:
show(query_60)

,city,COUNT(user_id)
0,Victoriastad,2
1,Reynoldsburgh,6
2,Kingville,7
3,Levybury,1
4,East Karenbury,2
...,...,...
38166,North Michaelaville,1
38167,Lake Kathystad,1
38168,East Cheyenneshire,1
38169,Lake Shari,1


---

## __7. HAVING__

61. Countries with more than 100 users.

In [21]:
table_schema["users"]

,Field,Type,Null,Key,Default,Extra
0,user_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,email,text,YES,,None,
3,phone,text,YES,,None,
4,city,text,YES,,None,
5,country,text,YES,,None,
6,created_at,datetime,YES,,None,


In [22]:
query_61 = """
    SELECT
        u.country,
        COUNT(u.user_id) 
    FROM
        users u
    GROUP BY
        u.country
    HAVING
        COUNT(u.user_id) > 100;
"""

In [23]:
show(query_61)

,country,COUNT(u.user_id)
0,Qatar,397
1,Estonia,376
2,Antigua and Barbuda,438
3,Italy,420
4,Saudi Arabia,375
...,...,...
238,Saint Pierre and Miquelon,419
239,Greece,400
240,Czech Republic,406
241,Christmas Island,395


-----

62. Categories with more than 50 products.

In [24]:
table_schema["products"]

,Field,Type,Null,Key,Default,Extra
0,product_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,category,text,YES,,None,
3,price,float,YES,,None,
4,stock,int,YES,,None,


In [27]:
query_62 = """
    SELECT
        p.category,
        COUNT(p.product_id)
    FROM
        products p
    GROUP BY
        p.category
    HAVING
        COUNT(p.product_id) > 50;
"""

In [28]:
show(query=query_62)

,category,COUNT(p.product_id)
0,Electronics,4050
1,Clothing,3961
2,Sports,3932
3,Books,4062
4,Home,3995


In [30]:
explain(query_62)

                                                                                                                                                                                                                                                                                                                                                    EXPLAIN
0  -> Filter: (`count(p.product_id)` > 50)  (actual time=37.4..37.4 rows=5 loops=1)\n    -> Table scan on <temporary>  (actual time=37.4..37.4 rows=5 loops=1)\n        -> Aggregate using temporary table  (actual time=37.3..37.3 rows=5 loops=1)\n            -> Table scan on p  (cost=2201 rows=21450) (actual time=0.0491..15.8 rows=20000 loops=1)\n


---

63. Users with more than 5 orders.

In [32]:
table_schema["order_items"]

,Field,Type,Null,Key,Default,Extra
0,order_item_id,varchar(36),NO,PRI,None,
1,order_id,varchar(36),YES,MUL,None,
2,product_id,varchar(36),YES,MUL,None,
3,quantity,int,YES,,None,


In [33]:
table_schema["users"]

,Field,Type,Null,Key,Default,Extra
0,user_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,email,text,YES,,None,
3,phone,text,YES,,None,
4,city,text,YES,,None,
5,country,text,YES,,None,
6,created_at,datetime,YES,,None,


In [34]:
table_schema["orders"]

,Field,Type,Null,Key,Default,Extra
0,order_id,varchar(36),NO,PRI,None,
1,user_id,varchar(36),YES,MUL,None,
2,order_date,datetime,YES,,None,
3,status,text,YES,,None,


In [28]:
query_63 = """
    SELECT
        o.user_id,
        COUNT(o.order_id)
    FROM
        orders o
    GROUP BY
        o.user_id
    HAVING
        COUNT(o.order_id) > 5;
"""

In [62]:
show(query_63)

,user_id,COUNT(o.order_id)
0,0008ea49-31dc-4361-bc18-896494715d1a,7
1,00180117-d71d-4607-8673-cd65dc85b2af,9
2,0018faac-b390-44c7-9203-63c232f57eb5,9
3,001994a8-e7eb-47e1-9076-85770471a1b6,8
4,00218624-8c6f-4e95-9441-10672a6b1552,8
...,...,...
8369,ffd0ed56-5afe-409e-83af-3a9fa3c4d53a,6
8370,ffe3a68f-2aea-474b-96e8-af3ea9a28552,6
8371,ffebc112-4009-43df-b395-ee9b49e02ae8,6
8372,ffeff44a-3fe9-4cf8-9002-fb024ca9b66d,7


In [63]:
explain(query_63)


==================== QUERY PLAN ====================

-> Filter: (count(o.order_id) > 5)  (cost=58130 rows=90954) (actual time=0.273..267 rows=8374 loops=1)
    -> Group aggregate: count(o.order_id), count(o.order_id)  (cost=58130 rows=90954) (actual time=0.249..260 rows=95057 loops=1)
        -> Covering index scan on o using user_id  (cost=30721 rows=274084) (actual time=0.239..133 rows=300000 loops=1)



---

64. Products sold more than 100 times.

In [64]:
table_schema["orders"]

,Field,Type,Null,Key,Default,Extra
0,order_id,varchar(36),NO,PRI,None,
1,user_id,varchar(36),YES,MUL,None,
2,order_date,datetime,YES,,None,
3,status,text,YES,,None,


In [20]:
table_schema["order_items"]

,Field,Type,Null,Key,Default,Extra
0,order_item_id,varchar(36),NO,PRI,None,
1,order_id,varchar(36),YES,MUL,None,
2,product_id,varchar(36),YES,MUL,None,
3,quantity,int,YES,,None,


In [21]:
query_64 = """
    SELECT 
        product_id, 
        SUM(quantity) AS total_sold
    FROM 
        order_items
    GROUP BY 
        product_id
    HAVING 
        total_sold > 100;
"""

In [23]:
show("""EXPLAIN SELECT product_id, SUM(quantity)
FROM order_items
GROUP BY product_id;""")

,id,select_type,table,partitions,type,possible_keys,key,key_len,ref,rows,filtered,Extra
0,1,SIMPLE,order_items,None,index,idx_order_items_product_id,idx_order_items_product_id,147,None,846665,100.0,None


In [24]:
explain(query_64)


==================== QUERY PLAN ====================

-> Filter: (total_sold > 100)  (cost=178252 rows=18355) (actual time=9.43..56068 rows=18893 loops=1)
    -> Group aggregate: sum(order_items.quantity)  (cost=178252 rows=18355) (actual time=9.42..56062 rows=20000 loops=1)
        -> Index scan on order_items using idx_order_items_product_id  (cost=93585 rows=846665) (actual time=9.39..55536 rows=899816 loops=1)



----

65. Payment methods with high total (>10000)

In [39]:
query_65 = """
    SELECT
        pa.payment_method,
        CAST(SUM(pa.amount) AS DECIMAL) total_payment
    FROM
        payments pa
    GROUP BY
        pa.payment_method
    HAVING
        total_payment > 10000;
"""

In [40]:
show(query_65)

,payment_method,total_payment
0,wallet,187960796
1,upi,187734018
2,card,189083039
3,netbanking,187131759


---

66. Cities with more than 50 users.

In [41]:
query_66 = """
    SELECT
        city,
        COUNT(u.user_id) AS user_count
    FROM
        users u
    GROUP BY
        u.city
    HAVING
        user_count > 50 ;
"""

In [42]:
show(query_66)

,city,user_count
0,North John,65
1,Port Michael,70
2,East John,66
3,East Jennifer,61
4,East Michael,72
5,East James,52
6,New David,54
7,North Michael,75
8,South David,53
9,Smithmouth,59


----

67. Categories with avg price > 1000.

In [43]:
table_schema["products"]

,Field,Type,Null,Key,Default,Extra
0,product_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,category,text,YES,,None,
3,price,float,YES,MUL,None,
4,stock,int,YES,,None,


In [44]:
query_67 = """
    SELECT
        p.category,
        AVG(p.price) Avg_price
    FROM
        products p
    GROUP BY
        p.category
    HAVING
        Avg_price > 1000;
"""

In [45]:
show(query_67)

,category,Avg_price
0,Electronics,2502.413220
1,Clothing,2500.986786
2,Sports,2518.002610
3,Books,2493.748187
4,Home,2514.534146


---

68. Users with total orders > 10

In [46]:
table_schema["orders"]

,Field,Type,Null,Key,Default,Extra
0,order_id,varchar(36),NO,PRI,None,
1,user_id,varchar(36),YES,MUL,None,
2,order_date,datetime,YES,MUL,None,
3,status,text,YES,,None,


In [48]:
query_68 = """
    SELECT
        o.user_id,
        COUNT(o.order_id) no_of_orders
    FROM
        orders o
    GROUP BY
        o.user_id
    HAVING
        no_of_orders > 10;
"""

In [49]:
show(query_68)

,user_id,no_of_orders
0,0be50a7b-2b5d-4021-9a0a-5e0ecd209a8d,13
1,0f09c761-e870-4013-9d87-884e99092ae8,11
2,14bcd49d-f2a8-49ad-b39c-8f86ea8cba3f,11
3,2061728d-d963-4a76-b95d-298645d8ec44,11
4,33631649-9f9e-4db6-a3fd-4c0b6f5660fc,12
5,44ae2546-3d45-4ad8-a59a-0eef912454f4,11
6,4802fa4b-76f8-468a-86e0-adbf1df1abc5,11
7,51b4aa6c-15de-4d23-96e9-7548a7452ca9,12
8,5caa47dd-5be8-4352-9510-fd7b1b034a9e,11
9,672de8d4-4204-4098-b6d2-bb5996a6905b,11


---

69. Products with low sales (<10).

In [51]:
order_items

,Field,Type,Null,Key,Default,Extra
0,order_item_id,varchar(36),NO,PRI,None,
1,order_id,varchar(36),YES,MUL,None,
2,product_id,varchar(36),YES,MUL,None,
3,quantity,int,YES,,None,


In [20]:
query_69 = """
    SELECT 
        product_id, 
        SUM(quantity) AS total_sold
    FROM 
        order_items
    GROUP BY 
        product_id
    HAVING 
        total_sold < 10;
"""

In [22]:
explain(query_69)


==================== QUERY PLAN ====================

-> Filter: (total_sold < 10)  (cost=180606 rows=19685) (actual time=366..366 rows=0 loops=1)
    -> Group aggregate: sum(order_items.quantity)  (cost=180606 rows=19685) (actual time=0.199..365 rows=20000 loops=1)
        -> Covering index scan on order_items using idx_product_quantity  (cost=95939 rows=846665) (actual time=0.15..140 rows=899816 loops=1)



In [23]:
show(query_69) # No matching rows

""


---

70. Countries with less than 20 users.

In [27]:
query_70 = """
    SELECT
        country,
        COUNT(u.user_id) user_count
    FROM
        users u
    GROUP BY
        u.country
    HAVING
        user_count > 20 ;
"""

In [28]:
show(query_70)

,country,user_count
0,Qatar,397
1,Estonia,376
2,Antigua and Barbuda,438
3,Italy,420
4,Saudi Arabia,375
...,...,...
238,Saint Pierre and Miquelon,419
239,Greece,400
240,Czech Republic,406
241,Christmas Island,395


-------

# __8. JOINs__


71. Join users and orders.

In [30]:
users

,Field,Type,Null,Key,Default,Extra
0,user_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,email,text,YES,,None,
3,phone,text,YES,,None,
4,city,text,YES,,None,
5,country,text,YES,,None,
6,created_at,datetime,YES,,None,


---

In [32]:
orders

,Field,Type,Null,Key,Default,Extra
0,order_id,varchar(36),NO,PRI,None,
1,user_id,varchar(36),YES,MUL,None,
2,order_date,datetime,YES,MUL,None,
3,status,text,YES,,None,


In [33]:
query_71 = """
    SELECT
        u.user_id,
        o.order_id
    FROM
        users u
    INNER JOIN
        orders o
    ON
        o.user_id = u.user_id;
"""

In [34]:
show(query_71)

,user_id,order_id
0,0000d817-629f-436b-b1e7-b71fe1102499,a7080e52-cea6-41d0-9d08-7ed5b83a0432
1,000196c5-99a4-4c4a-96ef-3ba48a564b85,4ad63b7e-0be3-418a-86fe-4fb2ee61611e
2,000196c5-99a4-4c4a-96ef-3ba48a564b85,eebaaf46-4485-4508-a3eb-c45d76d66294
3,00019ae4-fbf6-4589-840d-c142cbdcf552,ec429bc2-38fe-4134-b4db-31bf35287d24
4,00021d12-efcb-41a4-86b0-dd4b08f359d8,5ef3435a-33e8-498b-aca4-1a3a87a03628
...,...,...
299995,fffed7eb-c868-4014-96cb-a5e7b5a9f3b6,6a7b1056-35ca-42b7-a915-7e1749b6ad9e
299996,fffed7eb-c868-4014-96cb-a5e7b5a9f3b6,f4ae0152-1fa5-4079-a524-cfe8bc99298e
299997,fffee1a4-ea7b-4247-9247-3f0939fb0f5f,36a0da49-b458-44d3-ac86-c079e62c37e3
299998,fffee1a4-ea7b-4247-9247-3f0939fb0f5f,e026ab99-7f1c-4811-a5cc-b4dc96c3a223


---

72. Join orders and payments.

In [35]:
payments

,Field,Type,Null,Key,Default,Extra
0,payment_id,varchar(36),NO,PRI,None,
1,order_id,varchar(36),YES,MUL,None,
2,amount,float,YES,,None,
3,payment_method,text,YES,,None,
4,payment_status,text,YES,,None,
5,payment_date,datetime,YES,,None,


In [66]:
query_72 = """
    SELECT
        p.payment_id,
        o.order_id
    FROM
        orders o
    INNER JOIN
        payments p
    ON
        p.order_id = o.order_id
    LIMIT 20 ;
"""

In [69]:
show(query_72)

,payment_id,order_id
0,5b7e0ccf-9b25-4df1-ba14-483c3db239cb,ff27a74c-9d7c-4bab-9f8a-29ee7f4e7f82
1,7edc717a-b4ae-4202-bfa5-407351080e69,908becb9-6eb4-4c5a-a9bd-bf6b093c64d8
2,b8a10654-e471-4f6a-8610-a0195adebf5f,0236786a-50f6-4538-b576-8ce977777335
3,c90a068a-94f9-4735-80f1-13c88a9f2e10,9b503065-b134-4742-80c5-a22dfddf4e42
4,581c1225-67b7-4229-bf50-1104ebfbfc7a,b8048963-b292-4450-819f-a03b96cd673f
5,090168be-d874-40ca-8e03-08620ee5a980,c096d9dd-85e7-4bd7-9732-31b5f2ea0e8b
6,f9ee3f4b-0576-4bc1-8965-283fb9a7447f,6407b2cd-7cb3-494d-9210-942dbbb5bd4a
7,017d7c98-4018-4fef-8b70-c12eee0e4b28,aeb74038-3122-4408-a415-de51e0163e2a
8,bf6c46e5-7310-4de0-96c1-4f96bc9f8084,9534ded2-39e3-4455-8b12-524babe15967
9,a37a27e0-2675-4f97-b7d2-bd184c426540,c75da258-2f3a-484a-9acb-150c8f3571cb


---

73. Join order_items and products.

In [70]:
query_73 = """
    SELECT
        o_i.product_id,
        o_i.order_item_id
    FROM
        order_items o_i
    INNER JOIN
        products p
    ON
        o_i.product_id = p.product_id
    LIMIT 20;
"""

In [71]:
show(query_73)

,product_id,order_item_id
0,266326e5-6492-4bc3-988c-f7c729f68211,0be0fdf5-fb1b-40c5-9488-6790638037c6
1,266326e5-6492-4bc3-988c-f7c729f68211,1c1473f1-02ef-420c-8db0-9d520daea5b4
2,266326e5-6492-4bc3-988c-f7c729f68211,4155763d-951f-4d44-9162-3030cd75e749
3,266326e5-6492-4bc3-988c-f7c729f68211,43ba4d43-67c8-449f-8c93-499a3bea295d
4,266326e5-6492-4bc3-988c-f7c729f68211,728b5ae9-b503-403e-a568-af7cfe855fa9
5,266326e5-6492-4bc3-988c-f7c729f68211,932c7623-00e3-4458-9a88-ef574f83a5e2
6,266326e5-6492-4bc3-988c-f7c729f68211,973c0d55-b39e-4d5f-8098-7e5b9b691d62
7,266326e5-6492-4bc3-988c-f7c729f68211,e019e27e-37dd-4912-ae91-719adb4990a4
8,266326e5-6492-4bc3-988c-f7c729f68211,5de29ce0-f0c4-4699-926b-a6dd9ed7dd9a
9,266326e5-6492-4bc3-988c-f7c729f68211,5f9e1490-e842-4727-aed4-0c58a3efb2f0


---

74. Get product names with order quantity.

In [72]:
products

,Field,Type,Null,Key,Default,Extra
0,product_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,category,text,YES,,None,
3,price,float,YES,MUL,None,
4,stock,int,YES,,None,


In [73]:
order_items

,Field,Type,Null,Key,Default,Extra
0,order_item_id,varchar(36),NO,PRI,None,
1,order_id,varchar(36),YES,MUL,None,
2,product_id,varchar(36),YES,MUL,None,
3,quantity,int,YES,,None,


In [74]:
query_74 = """
    SELECT
        p.name,
        o_i.quantity
    FROM
        products p
    INNER JOIN
        order_items o_i
    ON
        p.product_id = o_i.product_id
    LIMIT 20 ;
"""

In [75]:
show(query_74)

,name,quantity
0,Western,1
1,Western,1
2,Western,1
3,Western,1
4,Western,1
5,Western,1
6,Western,1
7,Western,1
8,Western,1
9,Western,1


---

75. Get user names with order status.

In [76]:
users

,Field,Type,Null,Key,Default,Extra
0,user_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,email,text,YES,,None,
3,phone,text,YES,,None,
4,city,text,YES,,None,
5,country,text,YES,,None,
6,created_at,datetime,YES,,None,


In [77]:
orders

,Field,Type,Null,Key,Default,Extra
0,order_id,varchar(36),NO,PRI,None,
1,user_id,varchar(36),YES,MUL,None,
2,order_date,datetime,YES,MUL,None,
3,status,text,YES,,None,


In [78]:
query_75 = """
    SELECT
        u.name,
        o.status
    FROM
        users u
    INNER JOIN
        orders o
    ON
        u.user_id = o.user_id
    LIMIT 20;
"""

In [79]:
show(query_75)

,name,status
0,Haley Logan,placed
1,Fred Sullivan,shipped
2,Amanda Sims,delivered
3,Brandon Coleman,shipped
4,John Martinez,cancelled
5,Melissa Duarte,cancelled
6,Catherine Hall,cancelled
7,Albert Ashley,cancelled
8,Megan Ramirez,shipped
9,Anna Carr,delivered


---

76. Get payment details with user names.

In [80]:
payments

,Field,Type,Null,Key,Default,Extra
0,payment_id,varchar(36),NO,PRI,None,
1,order_id,varchar(36),YES,MUL,None,
2,amount,float,YES,,None,
3,payment_method,text,YES,,None,
4,payment_status,text,YES,,None,
5,payment_date,datetime,YES,,None,


In [81]:
users

,Field,Type,Null,Key,Default,Extra
0,user_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,email,text,YES,,None,
3,phone,text,YES,,None,
4,city,text,YES,,None,
5,country,text,YES,,None,
6,created_at,datetime,YES,,None,


In [82]:
orders

,Field,Type,Null,Key,Default,Extra
0,order_id,varchar(36),NO,PRI,None,
1,user_id,varchar(36),YES,MUL,None,
2,order_date,datetime,YES,MUL,None,
3,status,text,YES,,None,


In [89]:
query_76 = """
    SELECT
        p.payment_id,
        p.payment_method,
        p.payment_status,
        u.name
    FROM
        payments p
    INNER JOIN
        orders o
    ON
        p.order_id = o.order_id
    INNER JOIN
        users u
    ON
        o.user_id = u.user_id
    LIMIT 20 ;
"""

In [90]:
show(query_76)

,payment_id,payment_method,payment_status,name
0,b9188f29-ad1f-4dcd-ba4d-950410bb3214,netbanking,success,Misty Jackson
1,d7a31f71-da02-4ef9-940d-b3fbd4bcd68b,netbanking,success,Michael Harris
2,f4402a54-0594-4ce8-9779-10a7df5b70e4,upi,pending,Michael Harris
3,ad83334e-08f2-416b-9acf-e61e898c9bef,card,pending,David Durham
4,69fde8e1-b731-4047-a9e5-5de174c75830,card,failed,Nicole Morales
5,cf3398fc-b883-4b5c-95ae-956415db6ffc,upi,pending,Jordan James
6,83ed0452-8c66-4c49-9206-18a189758fa6,wallet,pending,Chase Lyons
7,faf944da-a2d5-4aa5-91f8-0ee28b153dcf,upi,failed,Chase Lyons
8,feb8e8b8-7112-4cc8-aeba-0af037f98351,card,pending,Chase Lyons
9,f483a75f-0c02-4e32-b4ea-59f5722fe1df,wallet,success,Chase Lyons


----

77. Join all tables to show full order info.

In [117]:
products

,Field,Type,Null,Key,Default,Extra
0,product_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,category,text,YES,,None,
3,price,float,YES,MUL,None,
4,stock,int,YES,,None,


In [119]:
payments

,Field,Type,Null,Key,Default,Extra
0,payment_id,varchar(36),NO,PRI,None,
1,order_id,varchar(36),YES,MUL,None,
2,amount,float,YES,,None,
3,payment_method,text,YES,,None,
4,payment_status,text,YES,,None,
5,payment_date,datetime,YES,,None,


In [120]:
users

,Field,Type,Null,Key,Default,Extra
0,user_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,email,text,YES,,None,
3,phone,text,YES,,None,
4,city,text,YES,,None,
5,country,text,YES,,None,
6,created_at,datetime,YES,,None,


In [122]:
order_items

,Field,Type,Null,Key,Default,Extra
0,order_item_id,varchar(36),NO,PRI,None,
1,order_id,varchar(36),YES,MUL,None,
2,product_id,varchar(36),YES,MUL,None,
3,quantity,int,YES,,None,


In [123]:
orders

,Field,Type,Null,Key,Default,Extra
0,order_id,varchar(36),NO,PRI,None,
1,user_id,varchar(36),YES,MUL,None,
2,order_date,datetime,YES,MUL,None,
3,status,text,YES,,None,


In [126]:
query_77 = """
    SELECT
        o.order_id,
        pay.payment_method,
        o_i.quantity,
        u.name,
        pro.name
    FROM
        orders o
    INNER JOIN
        order_items o_i
    ON
        o.order_id = o_i.order_id
    INNER JOIN
        users u
    ON
        u.user_id = o.user_id
    INNER JOIN
        products pro
    ON
        pro.product_id = o_i.product_id
    INNER JOIN
        payments pay
    ON
        pay.order_id = o.order_id
    LIMIT 10;
    """

In [127]:
show(query_77)

,order_id,payment_method,quantity,name,name
0,a7080e52-cea6-41d0-9d08-7ed5b83a0432,netbanking,3,Misty Jackson,Receive
1,a7080e52-cea6-41d0-9d08-7ed5b83a0432,netbanking,1,Misty Jackson,Know
2,4ad63b7e-0be3-418a-86fe-4fb2ee61611e,netbanking,5,Michael Harris,Far
3,4ad63b7e-0be3-418a-86fe-4fb2ee61611e,netbanking,4,Michael Harris,Recognize
4,4ad63b7e-0be3-418a-86fe-4fb2ee61611e,netbanking,2,Michael Harris,Keep
5,4ad63b7e-0be3-418a-86fe-4fb2ee61611e,netbanking,5,Michael Harris,Country
6,4ad63b7e-0be3-418a-86fe-4fb2ee61611e,netbanking,2,Michael Harris,I
7,eebaaf46-4485-4508-a3eb-c45d76d66294,upi,5,Michael Harris,Avoid
8,eebaaf46-4485-4508-a3eb-c45d76d66294,upi,2,Michael Harris,Road
9,eebaaf46-4485-4508-a3eb-c45d76d66294,upi,4,Michael Harris,Career


---

78. Find users with no orders

In [128]:
users

,Field,Type,Null,Key,Default,Extra
0,user_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,email,text,YES,,None,
3,phone,text,YES,,None,
4,city,text,YES,,None,
5,country,text,YES,,None,
6,created_at,datetime,YES,,None,


In [129]:
orders

,Field,Type,Null,Key,Default,Extra
0,order_id,varchar(36),NO,PRI,None,
1,user_id,varchar(36),YES,MUL,None,
2,order_date,datetime,YES,MUL,None,
3,status,text,YES,,None,


In [134]:
query_78 = """
    SELECT
        u.user_id,
        u.name
    FROM
        users u
    LEFT JOIN
        orders o
    ON
        u.user_id = o.user_id
    LIMIT 20;
"""

In [135]:
show(query_78)

,user_id,name
0,0000d817-629f-436b-b1e7-b71fe1102499,Misty Jackson
1,000196c5-99a4-4c4a-96ef-3ba48a564b85,Michael Harris
2,000196c5-99a4-4c4a-96ef-3ba48a564b85,Michael Harris
3,00019ae4-fbf6-4589-840d-c142cbdcf552,David Durham
4,00021d12-efcb-41a4-86b0-dd4b08f359d8,Nicole Morales
5,0003fccd-3076-41ef-beb4-28ecebcf9971,Jordan James
6,000600eb-cc78-433c-a7d0-4347c81478cd,Chase Lyons
7,000600eb-cc78-433c-a7d0-4347c81478cd,Chase Lyons
8,000600eb-cc78-433c-a7d0-4347c81478cd,Chase Lyons
9,000600eb-cc78-433c-a7d0-4347c81478cd,Chase Lyons


---

79. Find products never ordered.

In [138]:
order_items

,Field,Type,Null,Key,Default,Extra
0,order_item_id,varchar(36),NO,PRI,None,
1,order_id,varchar(36),YES,MUL,None,
2,product_id,varchar(36),YES,MUL,None,
3,quantity,int,YES,,None,


In [166]:
query_79 = """
    SELECT
        p.product_id
    FROM
        products p
    LEFT JOIN
        order_items or_it
    ON
        p.product_id = or_it.product_id
    WHERE
        or_it.product_id IS NULL
    LIMIT 20;
"""

In [167]:
show(query_79)

""


---

80. Get orders with product names.

In [168]:
orders

,Field,Type,Null,Key,Default,Extra
0,order_id,varchar(36),NO,PRI,None,
1,user_id,varchar(36),YES,MUL,None,
2,order_date,datetime,YES,MUL,None,
3,status,text,YES,,None,


In [169]:
products

,Field,Type,Null,Key,Default,Extra
0,product_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,category,text,YES,,None,
3,price,float,YES,MUL,None,
4,stock,int,YES,,None,


In [180]:
order_items

,Field,Type,Null,Key,Default,Extra
0,order_item_id,varchar(36),NO,PRI,None,
1,order_id,varchar(36),YES,MUL,None,
2,product_id,varchar(36),YES,MUL,None,
3,quantity,int,YES,,None,


In [189]:
query_80 = """
    SELECT
        or_it.order_id,
        p.product_id
    FROM
        orders o
    INNER JOIN
        order_items or_it
    ON
        or_it.order_id = o.order_id
    INNER JOIN
        products p
    ON
        p.product_id = or_it.product_id
    LIMIT 20 ;
"""

In [191]:
show(query_80)

,order_id,product_id
0,ff27a74c-9d7c-4bab-9f8a-29ee7f4e7f82,9f10cb52-a141-4286-b97e-3262b3f7ec0c
1,ff27a74c-9d7c-4bab-9f8a-29ee7f4e7f82,8cfee05f-39e0-4174-acbb-042071aa019e
2,908becb9-6eb4-4c5a-a9bd-bf6b093c64d8,3143204d-567f-4fe2-b112-749529f2043a
3,908becb9-6eb4-4c5a-a9bd-bf6b093c64d8,6c21920d-6f8e-4447-a5c1-7f8af3e8ed3b
4,908becb9-6eb4-4c5a-a9bd-bf6b093c64d8,bc0f9550-e62b-4920-8e57-6c5428a91837
5,0236786a-50f6-4538-b576-8ce977777335,45623405-5eea-4079-8957-161beb6d625e
6,0236786a-50f6-4538-b576-8ce977777335,6794da89-ed29-4dd2-a930-711b371ee4bb
7,0236786a-50f6-4538-b576-8ce977777335,4c5cc1e7-bca8-4447-a55f-af7469b254af
8,0236786a-50f6-4538-b576-8ce977777335,af58f4e0-a51e-4b17-be58-af5327a13b19
9,0236786a-50f6-4538-b576-8ce977777335,f15638ae-3bd5-49e8-acb7-e7c820a1b99b


----

## __9. SUBQUERIES__

81. Products above avg price.

In [192]:
products

,Field,Type,Null,Key,Default,Extra
0,product_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,category,text,YES,,None,
3,price,float,YES,MUL,None,
4,stock,int,YES,,None,


In [195]:
query_81 = """
    SELECT
        p.product_id,
        p.price
    FROM
        products p
    WHERE
        p.price > (SELECT AVG(p.price) FROM products p)
    LIMIT 20;
"""

In [196]:
show(query_81)

,product_id,price
0,3e66802f-a618-478b-95f7-26ec203c1e99,2506.13
1,b4f3b801-8982-460b-9f38-bc29204a6532,2506.87
2,c7d3d7c8-002c-43a8-8e1f-ad6ccda8745a,2507.45
3,d6dda29c-a909-4008-9244-16cbc847d09c,2507.46
4,671a705a-f45a-4f8f-9853-26315d515594,2507.61
5,7f0695d5-65e7-4de8-ad7d-8c4c7f7582a6,2507.78
6,b33f8e23-c9f4-41a8-b075-053cfb64840e,2508.34
7,0d60f9e2-157c-4cbf-a640-89e5e88864b4,2508.45
8,0e110e4c-8019-4d32-9855-e8a76eea6737,2509.03
9,317b00cf-0888-4158-8ad4-014798cc20ad,2509.17


-----

82. Users with more orders than average.

In [197]:
users

,Field,Type,Null,Key,Default,Extra
0,user_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,email,text,YES,,None,
3,phone,text,YES,,None,
4,city,text,YES,,None,
5,country,text,YES,,None,
6,created_at,datetime,YES,,None,


In [198]:
orders

,Field,Type,Null,Key,Default,Extra
0,order_id,varchar(36),NO,PRI,None,
1,user_id,varchar(36),YES,MUL,None,
2,order_date,datetime,YES,MUL,None,
3,status,text,YES,,None,


In [21]:
query_82 = """
    SELECT 
        user_id
    FROM 
        orders
    GROUP BY 
        user_id
    HAVING COUNT(*) > (
        SELECT 
            AVG(order_count)
        FROM (
            SELECT COUNT(*) AS order_count
            FROM orders
            GROUP BY user_id
    ) as t
);
"""

In [220]:
show(query_82)

,user_id
0,000600eb-cc78-433c-a7d0-4347c81478cd
1,00085c61-99fb-4187-a6e9-cab16436a8d1
2,0008ea49-31dc-4361-bc18-896494715d1a
3,000941b2-1ed2-49d9-bb9b-b807fbb41a42
4,0009cbe8-910f-4929-a52f-e5f994bc0d0a
...,...
35224,fff58729-d3be-40b6-ae18-81641c708bc1
35225,fff98178-7018-417f-90ce-29e5fee2d70a
35226,fffa54a1-87a6-414e-9b94-915fdc593b5b
35227,fffb5578-20bf-4116-a4e4-944aa3b77051


---

83. Orders with amount > avg payment.

In [28]:
payments

,Field,Type,Null,Key,Default,Extra
0,payment_id,varchar(36),NO,PRI,None,
1,order_id,varchar(36),YES,MUL,None,
2,amount,float,YES,,None,
3,payment_method,text,YES,,None,
4,payment_status,text,YES,,None,
5,payment_date,datetime,YES,,None,


In [26]:
query_83 = """
    SELECT
        *
    FROM
        payments p
    WHERE
        p.amount > (
            SELECT
                AVG(p.amount)
            FROM
                payments p
        ) 
    LIMIT 20;
"""

In [27]:
show(query_83)

,payment_id,order_id,amount,payment_method,payment_status,payment_date
0,0000ce28-6059-41fe-807d-5dc6c602779f,d71d0ed5-4ee7-48cc-935b-95b8d83c2943,4987.84,card,pending,2026-01-01 06:42:21
1,0000da01-dadb-46d1-9f7b-40c627f8fc09,d34e6141-6770-491a-973a-3a686b94134d,3330.91,wallet,failed,2026-03-22 11:23:36
2,00012b70-cdee-4584-8b06-e324f07b21c2,c555a2f6-c024-4774-9db7-5f92ad00029a,4913.13,wallet,pending,2026-02-07 11:47:42
3,0001937d-13d4-4fff-a789-8d4a391b85f8,708958cd-792b-4b05-aa30-a51c7c0a2273,3652.65,upi,failed,2026-01-10 13:38:33
4,0001a327-1e6c-4c3c-922b-3f20cdb85fb9,9d791a81-6c9f-47ba-b9ac-b51599dace51,2966.02,wallet,failed,2026-01-22 10:40:06
5,00028c07-0448-4bda-9f20-a6081340f47f,990e7b94-37b1-4313-844f-79647e36c670,2883.60,netbanking,success,2026-02-09 17:14:10
6,0002f9b3-d23b-4b35-b65c-2ce1897d454d,03917740-b109-4445-9402-f356fb357a2a,3385.27,upi,success,2026-02-15 19:50:50
7,00034ff0-7c34-41d8-aab1-97cdb8975922,33794ba7-6069-4444-beec-2e717ac6cc1a,3225.45,netbanking,pending,2026-02-13 08:56:12
8,0003c01e-4817-465f-ac07-64909b90a974,3009e5a9-8201-481a-b379-cb7d4ded6a17,3496.57,netbanking,failed,2026-03-04 02:48:52
9,0004480f-2f59-4027-a3c3-0734740d5b91,21114f1b-7404-4979-908b-420165958970,4066.11,card,failed,2026-03-03 10:58:02


---

84. Products with highest price.

In [28]:
products

,Field,Type,Null,Key,Default,Extra
0,product_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,category,text,YES,,None,
3,price,float,YES,MUL,None,
4,stock,int,YES,,None,


In [35]:
query_84 = """
    SELECT
        pr.name,
        pr.price
    FROM
        products pr
    WHERE
        pr.price = (
            SELECT
                MAX(pr.price)
            FROM
                products pr
        )
    LIMIT 20;
"""

In [36]:
show(query_84)

,name,price
0,Civil,4999.98


---

85. Users who made highest payment.

In [37]:
payments

,Field,Type,Null,Key,Default,Extra
0,payment_id,varchar(36),NO,PRI,None,
1,order_id,varchar(36),YES,MUL,None,
2,amount,float,YES,,None,
3,payment_method,text,YES,,None,
4,payment_status,text,YES,,None,
5,payment_date,datetime,YES,,None,


In [38]:
users

,Field,Type,Null,Key,Default,Extra
0,user_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,email,text,YES,,None,
3,phone,text,YES,,None,
4,city,text,YES,,None,
5,country,text,YES,,None,
6,created_at,datetime,YES,,None,


In [39]:
orders

,Field,Type,Null,Key,Default,Extra
0,order_id,varchar(36),NO,PRI,None,
1,user_id,varchar(36),YES,MUL,None,
2,order_date,datetime,YES,MUL,None,
3,status,text,YES,,None,


In [40]:
order_items

,Field,Type,Null,Key,Default,Extra
0,order_item_id,varchar(36),NO,PRI,None,
1,order_id,varchar(36),YES,MUL,None,
2,product_id,varchar(36),YES,MUL,None,
3,quantity,int,YES,,None,


In [47]:
query_85 = """
    SELECT
        o.user_id
    FROM
        orders o
    INNER JOIN
        payments pa
    ON
        o.order_id = pa.order_id
    WHERE
        pa.amount = (
            SELECT
                MAX(pa2.amount)
            FROM
                payments pa2
        )
    LIMIT 20;
"""

In [48]:
show(query_85)

,user_id
0,518c4c91-06a3-434c-adcf-c48bc6b2cbdb


---

86. Orders with max quantity.

In [49]:
orders

,Field,Type,Null,Key,Default,Extra
0,order_id,varchar(36),NO,PRI,None,
1,user_id,varchar(36),YES,MUL,None,
2,order_date,datetime,YES,MUL,None,
3,status,text,YES,,None,


In [50]:
order_items

,Field,Type,Null,Key,Default,Extra
0,order_item_id,varchar(36),NO,PRI,None,
1,order_id,varchar(36),YES,MUL,None,
2,product_id,varchar(36),YES,MUL,None,
3,quantity,int,YES,,None,


In [59]:
query_86 = """
    SELECT
        or_it.order_id
    FROM
        order_items or_it
    WHERE
        or_it.quantity = (
            SELECT
                MAX(or_it2.quantity) as max_quantity
            FROM
                order_items or_it2
        )
    LIMIT 20;
"""

In [60]:
show(query_86)

,order_id
0,80a28a24-8985-43c8-ac12-8e8008a0c06f
1,ec616acc-449c-472c-842c-38a00567a763
2,2f91a65f-7911-4ca2-9b4c-7beaab7d6639
3,6fa631f9-1860-4295-abe5-e6af931226a8
4,7483a03d-a47d-43eb-8359-a7ecb1bd1b43
5,a0d67468-aa91-4f08-99fa-2a6d3e46efa2
6,d691224f-59de-4d38-851e-7d9664ef30b2
7,41909749-81a5-4e2e-85e6-62d3a156274c
8,8df70534-eaa2-4f95-88bd-00b338a9cfb8
9,34bfa7f1-0095-4de7-bee5-e37653a34c58


---

87. Categories with highest avg price.


In [21]:
products

,Field,Type,Null,Key,Default,Extra
0,product_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,category,text,YES,,None,
3,price,float,YES,MUL,None,
4,stock,int,YES,,None,


In [38]:
query_87 = """
    SELECT
        pr.category,
        AVG(pr.price)
    FROM
        products pr
    GROUP BY
        pr.category
    HAVING
       AVG(pr.price) = (
           SELECT
               MAX(avg_price)
            FROM
            (
                SELECT
                    AVG(pr2.price) AS avg_price
                FROM
                    products pr2
                GROUP BY
                    pr2.category
            ) as t
       )
    LIMIT 20;
"""

In [22]:
alternative = """SELECT
    category
FROM
    products
GROUP BY
    category
ORDER BY
    AVG(price) DESC
LIMIT 1;"""

In [39]:
show(query_87)

,category,AVG(pr.price)
0,Sports,2518.00261


----

88. Users with no orders.

In [20]:
users

,Field,Type,Null,Key,Default,Extra
0,user_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,email,text,YES,,None,
3,phone,text,YES,,None,
4,city,text,YES,,None,
5,country,text,YES,,None,
6,created_at,datetime,YES,,None,


In [21]:
orders

,Field,Type,Null,Key,Default,Extra
0,order_id,varchar(36),NO,PRI,None,
1,user_id,varchar(36),YES,MUL,None,
2,order_date,datetime,YES,MUL,None,
3,status,text,YES,,None,


In [27]:
query_88 = """
    SELECT
        *
    FROM
        users u
    WHERE
        u.user_id NOT IN (
            SELECT
                o.user_id
            FROM
                orders o
        )
    LIMIT 20 ;
"""

In [28]:
show(query_88)

,user_id,name,email,phone,city,country,created_at
0,001eac44-adfb-4d59-b18c-4110ae5a7c96,Victoria Lawrence,hannahsullivan@example.com,(930)210-9876x8641,West Tony,Greenland,2021-09-10 18:25:17
1,004afe34-d065-499e-8370-22dc727181a2,Jeremy Parrish,brianna22@example.org,963-343-8525x314,New Morganville,Guyana,2025-07-11 22:41:53
2,005730ae-0d94-4753-b4af-21a4d1e1a8bc,John Cox,paceanthony@example.net,317.592.5657,South Ashley,Djibouti,2022-03-29 01:24:52
3,0064dbba-88f2-4a83-aac2-b176feb2bc95,Brian Serrano,kristina52@example.org,374.642.2203,North Jill,Bosnia and Herzegovina,2024-12-27 04:53:56
4,0064ecd9-a552-40c0-9f66-1bc844deb16e,Dennis Lynch,carla57@example.org,+1-917-243-6112x67583,Kimshire,China,2023-02-02 10:20:23
5,00693a91-184b-49d1-b535-5f96a62316ca,Kyle Gray,campbellandrea@example.org,+1-782-338-9387x7131,Cannonton,Russian Federation,2023-10-13 09:36:43
6,00695c7e-b5b4-4006-8c72-4152fd9e9416,Kevin Diaz,crossbrian@example.net,001-568-281-1370x8692,West Duane,Ireland,2023-01-26 12:54:13
7,0074f838-cfb8-4efe-8d1a-3e4e8f9724bb,Vincent Jenkins,katherine06@example.com,8013761887,West Sara,Colombia,2020-04-11 06:03:51
8,007e3157-6e4a-4210-bd4e-d26b962a30b7,Dakota Gamble,gomezwilliam@example.com,489-753-8271x571,Espinozashire,Lebanon,2021-01-16 12:41:21
9,008f825a-c667-40b4-a031-999fa717462c,Christopher Reyes,stevencontreras@example.net,378.605.0601x7236,Marshallburgh,Tajikistan,2020-04-20 04:09:49


> Why the following alternative   

> Handles NULL values safely ✅   
> More reliable across databases ✅   
> Often performs better with indexes ✅  

In [30]:
alternative1 = """
SELECT
    *
FROM
    users u
WHERE
    NOT EXISTS (
        SELECT 1
        FROM orders o
        WHERE o.user_id = u.user_id
    )
LIMIT 20;
"""

> Why the following alternative is better     
> Simpler

In [31]:
alternative = """SELECT
    u.*
FROM
    users u
LEFT JOIN orders o
    ON u.user_id = o.user_id
WHERE
    o.user_id IS NULL
LIMIT 20;"""

---

89. Products never sold.

In [23]:
products

,Field,Type,Null,Key,Default,Extra
0,product_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,category,text,YES,,None,
3,price,float,YES,MUL,None,
4,stock,int,YES,,None,


In [31]:
order_items

,Field,Type,Null,Key,Default,Extra
0,order_item_id,varchar(36),NO,PRI,None,
1,order_id,varchar(36),YES,MUL,None,
2,product_id,varchar(36),YES,MUL,None,
3,quantity,int,YES,,None,


In [25]:
orders

,Field,Type,Null,Key,Default,Extra
0,order_id,varchar(36),NO,PRI,None,
1,user_id,varchar(36),YES,MUL,None,
2,order_date,datetime,YES,MUL,None,
3,status,text,YES,,None,


In [56]:
query_89 = """
    SELECT
        *
    FROM
        products pr
    WHERE
        NOT EXISTS (
            SELECT
                1
            FROM
                order_items or_it
            WHERE
                pr.product_id = or_it.product_id
        )
    LIMIT 20;
"""

In [57]:
show(query_89)

""


---

90. Payments above avg per user.

In [58]:
payments

,Field,Type,Null,Key,Default,Extra
0,payment_id,varchar(36),NO,PRI,None,
1,order_id,varchar(36),YES,MUL,None,
2,amount,float,YES,,None,
3,payment_method,text,YES,,None,
4,payment_status,text,YES,,None,
5,payment_date,datetime,YES,,None,


In [59]:
users

,Field,Type,Null,Key,Default,Extra
0,user_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,email,text,YES,,None,
3,phone,text,YES,,None,
4,city,text,YES,,None,
5,country,text,YES,,None,
6,created_at,datetime,YES,,None,


In [27]:
orders

,Field,Type,Null,Key,Default,Extra
0,order_id,varchar(36),NO,PRI,None,
1,user_id,varchar(36),YES,MUL,None,
2,order_date,datetime,YES,MUL,None,
3,status,text,YES,,None,


In [37]:
query_90 = """
    SELECT 
        * 
    FROM 
        payments p1
    WHERE 
        amount > (
            SELECT 
                AVG(amount)
            FROM
                payments p2
            WHERE 
                p1.order_id = p2.order_id
        )
    LIMIT 20;
"""

In [41]:
show(query_90)

""


---

## __10. CASE WHEN__ 

91. Label products as 'Expensive' (>1000) else 'Cheap'.

In [42]:
products

,Field,Type,Null,Key,Default,Extra
0,product_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,category,text,YES,,None,
3,price,float,YES,MUL,None,
4,stock,int,YES,,None,


In [45]:
query_91 = """
    SELECT
        pr.product_id,
        pr.name,
        CASE
            WHEN pr.price > 1000 THEN "Expensive"
            ELSE "Cheap"
        END AS price_label
    FROM
        products pr
    LIMIT 20;
"""

In [46]:
show(query_91)

,product_id,name,price_label
0,0003c4a5-212c-47ef-af0f-cdee99c413c5,Western,Cheap
1,0006cdee-589d-4643-bc5b-a4706b439d5a,Cup,Expensive
2,00081441-ccf7-48a3-ba59-c3552457b66b,Mouth,Cheap
3,00175b94-a20d-4273-8b80-edaf89168774,Stay,Cheap
4,001a274d-067e-4cc2-92f7-7521cb51ed43,Within,Cheap
5,0020fd29-307b-45ee-b625-ddfda55b91bd,Project,Expensive
6,0022a06a-d276-490b-beb4-93e747b5b8c9,Sound,Expensive
7,00238654-2aea-47c4-afe8-6f1bcae16246,Decade,Expensive
8,00245323-f4ee-44c0-b518-80dc50ba906f,Deal,Expensive
9,00246097-550b-4c37-912d-a884d8ffa959,War,Expensive


----

92. Categorize users by country.

In [47]:
query_92 = """
    SELECT
        user_id,
        name,
        CASE
            WHEN u.country = "India" THEN "Local"
            ELSE "International"
        END AS nationality
    FROM
        users u
    LIMIT 20;
"""

In [48]:
show(query_92)

,user_id,name,nationality
0,0000d817-629f-436b-b1e7-b71fe1102499,Misty Jackson,International
1,000196c5-99a4-4c4a-96ef-3ba48a564b85,Michael Harris,International
2,00019ae4-fbf6-4589-840d-c142cbdcf552,David Durham,International
3,00021d12-efcb-41a4-86b0-dd4b08f359d8,Nicole Morales,International
4,0003fccd-3076-41ef-beb4-28ecebcf9971,Jordan James,International
5,000600eb-cc78-433c-a7d0-4347c81478cd,Chase Lyons,International
6,000614d8-ab87-48a4-825b-a319b53c3abe,Andrew Hurst,International
7,000669b2-a8d5-45e7-8a49-313cd6a3352f,David Petersen,International
8,000685e8-a046-42b4-bda2-99faf6f2dfb8,Cindy Ward,International
9,0006bba7-2638-4a9e-94cc-46404f4dc955,Timothy Walker,International


---

93. Label payments as 'High' (>2000) or 'Low'.

In [49]:
query_93 = """
    SELECT
        pa.payment_id,
        pa.amount,
        CASE
            WHEN pa.amount > 2000 THEN "High"
            ELSE "low"
        END AS "category"
    FROM
        payments pa
    LIMIT 20;
"""

In [50]:
show(query_93)

,payment_id,amount,category
0,0000803a-bfb4-4dfa-81fe-10884fd5ae37,1806.55,low
1,0000cccd-df3a-4a79-9b6d-c3efe1645b8b,74.54,low
2,0000ce28-6059-41fe-807d-5dc6c602779f,4987.84,High
3,0000da01-dadb-46d1-9f7b-40c627f8fc09,3330.91,High
4,0000eefa-e994-4138-ac26-75409bc0977c,725.73,low
5,00012b70-cdee-4584-8b06-e324f07b21c2,4913.13,High
6,000143ae-f1b7-4981-a69f-2debe8281861,483.09,low
7,00014e00-2c00-444f-8b5b-4e57927ce4b2,1305.83,low
8,0001937d-13d4-4fff-a789-8d4a391b85f8,3652.65,High
9,0001a327-1e6c-4c3c-922b-3f20cdb85fb9,2966.02,High


---

94. Mark orders as 'Completed' or 'Pending'.

In [51]:
orders

,Field,Type,Null,Key,Default,Extra
0,order_id,varchar(36),NO,PRI,None,
1,user_id,varchar(36),YES,MUL,None,
2,order_date,datetime,YES,MUL,None,
3,status,text,YES,,None,


In [52]:
show("select distinct status from orders ;")

,status
0,placed
1,shipped
2,delivered
3,cancelled


In [57]:
query_94 = """
    SELECT
        o.order_id,
        o.status,
        CASE
            WHEN o.status = "delivered" THEN "Completed"
            ELSE "pending"
        END AS is_completed
    FROM
        orders o
    LIMIT 20;
"""

In [58]:
show(query_94)

,order_id,status,is_completed
0,000015f8-ff52-41a7-bd5a-0f9eb1cd082e,placed,pending
1,00003c7e-dd5b-4e7b-bae1-9384b38b1b46,shipped,pending
2,00005f1a-f312-4ddb-9158-89215fcc79d1,delivered,Completed
3,0000f575-ff08-4bd2-8006-c4aa34f569c3,shipped,pending
4,0001110c-0a0c-4b1f-a029-bae65044b838,cancelled,pending
5,000192e5-ea02-4cb8-9d89-7ec3bd1d2ad3,cancelled,pending
6,0001c4e3-0211-41c5-bc5d-7d6d2860073d,cancelled,pending
7,0001d9dc-ecf6-454c-b0a8-b0a1358d737d,cancelled,pending
8,000221ed-a7a1-42f4-ac1f-e2870f1bd147,shipped,pending
9,00028802-8fad-44c3-b074-db650331fab3,delivered,Completed


---

95. Categorize stock as 'Low', 'Medium', 'High'.

In [59]:
products

,Field,Type,Null,Key,Default,Extra
0,product_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,category,text,YES,,None,
3,price,float,YES,MUL,None,
4,stock,int,YES,,None,


In [63]:
preview("products")

,product_id,name,category,price,stock
0,0003c4a5-212c-47ef-af0f-cdee99c413c5,Western,Electronics,352.58,819
1,0006cdee-589d-4643-bc5b-a4706b439d5a,Cup,Clothing,4713.70,872
2,00081441-ccf7-48a3-ba59-c3552457b66b,Mouth,Sports,74.22,83
3,00175b94-a20d-4273-8b80-edaf89168774,Stay,Electronics,684.80,955
4,001a274d-067e-4cc2-92f7-7521cb51ed43,Within,Books,849.07,892
5,0020fd29-307b-45ee-b625-ddfda55b91bd,Project,Electronics,3634.29,602
6,0022a06a-d276-490b-beb4-93e747b5b8c9,Sound,Sports,1447.02,851
7,00238654-2aea-47c4-afe8-6f1bcae16246,Decade,Home,3449.32,235
8,00245323-f4ee-44c0-b518-80dc50ba906f,Deal,Clothing,2672.84,310
9,00246097-550b-4c37-912d-a884d8ffa959,War,Sports,1613.24,690


In [65]:
query_95 = """
    SELECT
        pr.product_id,
        pr.name,
        CASE
            WHEN pr.stock >= 0 AND pr.stock <= 350 THEN "Low"
            WHEN pr.stock > 350 AND pr.stock <= 660 THEN "Medium"
            ELSE "High"
        END AS stock_category
    FROM
        products pr
    LIMIT 20;
"""

In [66]:
show(query_95)

,product_id,name,stock_category
0,0003c4a5-212c-47ef-af0f-cdee99c413c5,Western,High
1,0006cdee-589d-4643-bc5b-a4706b439d5a,Cup,High
2,00081441-ccf7-48a3-ba59-c3552457b66b,Mouth,Low
3,00175b94-a20d-4273-8b80-edaf89168774,Stay,High
4,001a274d-067e-4cc2-92f7-7521cb51ed43,Within,High
5,0020fd29-307b-45ee-b625-ddfda55b91bd,Project,Medium
6,0022a06a-d276-490b-beb4-93e747b5b8c9,Sound,High
7,00238654-2aea-47c4-afe8-6f1bcae16246,Decade,Low
8,00245323-f4ee-44c0-b518-80dc50ba906f,Deal,Low
9,00246097-550b-4c37-912d-a884d8ffa959,War,High


---

96. Label users as 'Active' if orders > 5.

In [67]:
order_items

,Field,Type,Null,Key,Default,Extra
0,order_item_id,varchar(36),NO,PRI,None,
1,order_id,varchar(36),YES,MUL,None,
2,product_id,varchar(36),YES,MUL,None,
3,quantity,int,YES,,None,


In [68]:
orders

,Field,Type,Null,Key,Default,Extra
0,order_id,varchar(36),NO,PRI,None,
1,user_id,varchar(36),YES,MUL,None,
2,order_date,datetime,YES,MUL,None,
3,status,text,YES,,None,


In [69]:
query_96 = """
    SELECT
        o.user_id,
        CASE
            WHEN COUNT(order_id) > 5 THEN "Active"
            ELSE "Inactive"
        END AS active_status
    FROM
        orders o
    GROUP BY
        o.user_id
    LIMIT 20;
"""

In [70]:
show(query_96)

,user_id,active_status
0,0000d817-629f-436b-b1e7-b71fe1102499,Inactive
1,000196c5-99a4-4c4a-96ef-3ba48a564b85,Inactive
2,00019ae4-fbf6-4589-840d-c142cbdcf552,Inactive
3,00021d12-efcb-41a4-86b0-dd4b08f359d8,Inactive
4,0003fccd-3076-41ef-beb4-28ecebcf9971,Inactive
5,000600eb-cc78-433c-a7d0-4347c81478cd,Inactive
6,000614d8-ab87-48a4-825b-a319b53c3abe,Inactive
7,000669b2-a8d5-45e7-8a49-313cd6a3352f,Inactive
8,000685e8-a046-42b4-bda2-99faf6f2dfb8,Inactive
9,0006bba7-2638-4a9e-94cc-46404f4dc955,Inactive


---

97. Categorize categories based on avg price.


In [71]:
products

,Field,Type,Null,Key,Default,Extra
0,product_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,category,text,YES,,None,
3,price,float,YES,MUL,None,
4,stock,int,YES,,None,


In [84]:
query_97 = """
    SELECT
        pr.category,
        AVG(pr.price),
        CASE
            WHEN AVG(pr.price) > 2500 THEN "High Average"
            WHEN AVG(pr.price) = 2500 THEN "Normal"
            ELSE "Low Average"
        END AS category_of_category
    FROM
        products pr
    GROUP BY
        pr.category
    LIMIT 20;
"""

In [85]:
show(query_97)

,category,AVG(pr.price),category_of_category
0,Electronics,2502.413220,High Average
1,Clothing,2500.986786,High Average
2,Sports,2518.002610,High Average
3,Books,2493.748187,Low Average
4,Home,2514.534146,High Average


----

98. Label payments as success/failure group.

In [86]:
payments

,Field,Type,Null,Key,Default,Extra
0,payment_id,varchar(36),NO,PRI,None,
1,order_id,varchar(36),YES,MUL,None,
2,amount,float,YES,,None,
3,payment_method,text,YES,,None,
4,payment_status,text,YES,,None,
5,payment_date,datetime,YES,,None,


In [87]:
preview("payments")

,payment_id,order_id,amount,payment_method,payment_status,payment_date
0,0000803a-bfb4-4dfa-81fe-10884fd5ae37,6799a061-d1bb-41da-8e80-8eeb76fe7f5f,1806.55,wallet,success,2026-01-10 01:23:35
1,0000cccd-df3a-4a79-9b6d-c3efe1645b8b,1b00e2f1-7377-4e26-90c4-9c64707f14a5,74.54,upi,pending,2026-03-10 20:23:57
2,0000ce28-6059-41fe-807d-5dc6c602779f,d71d0ed5-4ee7-48cc-935b-95b8d83c2943,4987.84,card,pending,2026-01-01 06:42:21
3,0000da01-dadb-46d1-9f7b-40c627f8fc09,d34e6141-6770-491a-973a-3a686b94134d,3330.91,wallet,failed,2026-03-22 11:23:36
4,0000eefa-e994-4138-ac26-75409bc0977c,a8f63aac-8002-4682-90d8-48c903e0b87b,725.73,netbanking,success,2026-03-19 23:40:23
5,00012b70-cdee-4584-8b06-e324f07b21c2,c555a2f6-c024-4774-9db7-5f92ad00029a,4913.13,wallet,pending,2026-02-07 11:47:42
6,000143ae-f1b7-4981-a69f-2debe8281861,45c17096-daa4-4622-9940-f85f99aca178,483.09,netbanking,success,2026-03-22 21:20:17
7,00014e00-2c00-444f-8b5b-4e57927ce4b2,776dc260-2c72-41b9-8c46-8944fc680a68,1305.83,netbanking,success,2026-03-09 14:57:08
8,0001937d-13d4-4fff-a789-8d4a391b85f8,708958cd-792b-4b05-aa30-a51c7c0a2273,3652.65,upi,failed,2026-01-10 13:38:33
9,0001a327-1e6c-4c3c-922b-3f20cdb85fb9,9d791a81-6c9f-47ba-b9ac-b51599dace51,2966.02,wallet,failed,2026-01-22 10:40:06


In [91]:
query_98 = """
    SELECT
        pa.payment_id,
        -- pa.payment_status,
        CASE
            WHEN pa.payment_status = "success" THEN "success"
            ELSE "failure"
        END AS status
    FROM
        payments pa
    LIMIT 20;
"""

In [92]:
show(query_98)

,payment_id,status
0,0000803a-bfb4-4dfa-81fe-10884fd5ae37,success
1,0000cccd-df3a-4a79-9b6d-c3efe1645b8b,failure
2,0000ce28-6059-41fe-807d-5dc6c602779f,failure
3,0000da01-dadb-46d1-9f7b-40c627f8fc09,failure
4,0000eefa-e994-4138-ac26-75409bc0977c,success
5,00012b70-cdee-4584-8b06-e324f07b21c2,failure
6,000143ae-f1b7-4981-a69f-2debe8281861,success
7,00014e00-2c00-444f-8b5b-4e57927ce4b2,success
8,0001937d-13d4-4fff-a789-8d4a391b85f8,failure
9,0001a327-1e6c-4c3c-922b-3f20cdb85fb9,failure


----

99. Create order size category (based on items).

In [97]:
order_items

,Field,Type,Null,Key,Default,Extra
0,order_item_id,varchar(36),NO,PRI,None,
1,order_id,varchar(36),YES,MUL,None,
2,product_id,varchar(36),YES,MUL,None,
3,quantity,int,YES,,None,


In [100]:
preview("order_items")

,order_item_id,order_id,product_id,quantity
0,00000dd0-97cc-44fd-9538-3efc117f8219,88a718c3-0a1f-4aa6-bd18-87f93b6e2e87,d5841655-4998-4bac-b2d5-cd573140b43f,2
1,00003118-eb66-47bc-9dc0-12111686ec6e,80a28a24-8985-43c8-ac12-8e8008a0c06f,52183a16-e605-461c-a5eb-4a9489f28753,5
2,00003653-a179-4879-839c-cd9cd72b9761,71007dc3-17e0-4c06-8d29-10de762f6afa,796a2f4f-c007-4990-b315-5ba9f8dc8981,4
3,00004c49-1691-45c0-8cc2-30f068615d80,b82a34c4-3ecc-4c5e-bbff-075a8416884d,13eaa824-1327-4135-a321-5650531746da,2
4,00005156-71cd-489b-8744-93ebfc52cda7,243ef973-080f-4bbc-bb0f-b028f50e3164,8733590d-d605-4c6f-9da8-3be86aa1ae89,1
5,000057d7-9611-4752-bd12-9ecf708311ad,59cf0fe7-1e78-403b-a96b-ecc63ecc2c57,6de33771-d42c-431c-9b92-4b24a0017e4a,1
6,00005cde-675d-4d4d-99d9-4dfb109154fe,3443426f-2089-42b5-b61f-460b69dbfab4,3e3de76c-c3b4-4b45-a95a-c5ce6f85acf9,1
7,0000857a-57c6-490d-b283-852a93976f79,0b3a4ee3-7864-4ba3-9585-aa819290c597,acc16f2e-15aa-4cb7-a28a-b3489336275e,2
8,0000bd08-7b2e-43fd-90dc-1434fd5b30f6,98af6ec0-f7d1-40f7-8269-55aa50ade9ca,444f07dc-3523-4f07-9e17-6c232eb0faf6,2
9,0000dc26-ad65-4c51-822c-86049576d5a8,d0c54dd1-d675-48ee-b865-b28e40668e03,122439b4-1207-46ae-94f1-28eacc7aeea0,4


In [101]:
query_99 = """
    SELECT
        order_id,
        CASE
            WHEN SUM(or_it.quantity) > 5 THEN "Bulk Order"
            ELSE "Normal Order"
        END AS order_size
    FROM
        order_items or_it
    GROUP BY
        order_id
    LIMIT 20 ;
"""

In [102]:
show(query_99)

,order_id,order_size
0,000015f8-ff52-41a7-bd5a-0f9eb1cd082e,Bulk Order
1,00003c7e-dd5b-4e7b-bae1-9384b38b1b46,Bulk Order
2,00005f1a-f312-4ddb-9158-89215fcc79d1,Normal Order
3,0000f575-ff08-4bd2-8006-c4aa34f569c3,Bulk Order
4,0001110c-0a0c-4b1f-a029-bae65044b838,Normal Order
5,000192e5-ea02-4cb8-9d89-7ec3bd1d2ad3,Bulk Order
6,0001c4e3-0211-41c5-bc5d-7d6d2860073d,Bulk Order
7,0001d9dc-ecf6-454c-b0a8-b0a1358d737d,Normal Order
8,000221ed-a7a1-42f4-ac1f-e2870f1bd147,Bulk Order
9,00028802-8fad-44c3-b074-db650331fab3,Bulk Order


---

100. Categorize cities as Tier 1 / Tier 2.

In [111]:
query_100 = """
    SELECT
        u.city,
        CASE
            WHEN COUNT(u.user_id) > 5 THEN "Tier 1"
            ELSE "Tier 2"
        END AS city_type
    FROM
        users u
    GROUP BY
        u.city
    LIMIT 20;
"""

In [112]:
show(query_100)

,city,city_type
0,Victoriastad,Tier 2
1,Reynoldsburgh,Tier 1
2,Kingville,Tier 1
3,Levybury,Tier 2
4,East Karenbury,Tier 2
5,East Evan,Tier 2
6,East Ronald,Tier 1
7,Gilbertshire,Tier 2
8,Rossport,Tier 2
9,East Julieview,Tier 2


---

## __11. WINDOW FUNCTIONS__

101. Rank products by price.

In [42]:
query_101 = """
    SELECT
        pr.product_id,
        pr.price,
        RANK() OVER (ORDER BY pr.price DESC) AS ranks    
    FROM
        products pr
    LIMIT 20;
"""

In [43]:
show(query_101)

,product_id,price,ranks
0,c5edb425-bda5-4f14-86e4-29b3f609fc10,4999.98,1
1,882fab09-1abc-49e0-a7db-0f1681f2dbb1,4999.97,2
2,db40bc48-328f-4eb5-8a65-bb0dec061dfb,4999.71,3
3,1be71191-46b0-4a04-8b72-0841a922fc25,4999.53,4
4,4a199d88-d324-4d5d-a275-2ec2ce0e2655,4999.43,5
5,8befcd10-4525-46e2-aafa-ca71936855a2,4999.31,6
6,0f4b500f-d323-407a-a540-b913e6dac977,4999.20,7
7,f61fe562-9026-4c84-826e-7862bb4fed9e,4999.07,8
8,196df4f2-e6c6-4837-8b2e-ee510b0a5d45,4998.17,9
9,f8d0615e-6b2c-4b91-a3db-8cf58b37732d,4997.95,10


----

102. Row number for orders per user.

In [44]:
orders

,Field,Type,Null,Key,Default,Extra
0,order_id,varchar(36),NO,PRI,None,
1,user_id,varchar(36),YES,MUL,None,
2,order_date,datetime,YES,MUL,None,
3,status,text,YES,,None,


In [51]:
query_102 = """
    SELECT
        o.order_id,
        ROW_NUMBER() OVER(PARTITION BY o.user_id ORDER BY o.order_id DESC) AS roww
    FROM
        orders o
    LIMIT 20;
"""

In [52]:
show(query_102)

,order_id,roww
0,a7080e52-cea6-41d0-9d08-7ed5b83a0432,1
1,eebaaf46-4485-4508-a3eb-c45d76d66294,1
2,4ad63b7e-0be3-418a-86fe-4fb2ee61611e,2
3,ec429bc2-38fe-4134-b4db-31bf35287d24,1
4,5ef3435a-33e8-498b-aca4-1a3a87a03628,1
5,32340def-890b-4410-b195-9b91fbf06992,1
6,bfbdf1f1-fa9b-40d0-b08d-12ab57fe2f3a,1
7,ad156c4b-f7fb-430b-94cb-7a8d420cd856,2
8,74ad48c1-003a-4d26-a1c8-e0d872315c31,3
9,1bf0baea-9808-4155-99d9-864907635007,4


---

103. Running total of payments.

In [22]:
payments

,Field,Type,Null,Key,Default,Extra
0,payment_id,varchar(36),NO,PRI,None,
1,order_id,varchar(36),YES,MUL,None,
2,amount,float,YES,,None,
3,payment_method,text,YES,,None,
4,payment_status,text,YES,,None,
5,payment_date,datetime,YES,,None,


In [27]:
query_103 = """
    SELECT
        pa.payment_id,
        pa.amount,
        SUM(pa.amount) OVER (ORDER BY pa.payment_date) AS running_total
    FROM
        payments pa
    LIMIT 20;
"""

In [28]:
show(query_103)

,payment_id,amount,running_total
0,8cc2f606-73aa-40ac-8291-af6cc6240511,2126.13,2126.129883
1,07b247bc-5b84-475f-8641-365c11defd00,2148.34,4274.469971
2,821ca84c-f14b-43f5-85d6-71436ef27928,1576.66,5851.130005
3,b0e1b4a7-ba48-4024-8afd-31ddff765aa0,4214.58,10065.710083
4,50e31a28-eace-436d-8b39-48b6c64c9bec,4522.18,14587.890259
5,5fc5fffa-9b98-478c-9794-68670c8044e8,4332.88,18920.770142
6,bae42a6c-1451-46ed-87bb-01dacda6de87,4575.35,23496.120239
7,240a8bb7-34c7-4486-b6b0-af499e9b4ea1,4836.12,28332.240356
8,6cb715b6-be76-4a3c-b0b6-2696e4039128,3979.28,32311.520386
9,fd3e03c8-cc70-484d-a8af-b6049f828e4c,2229.97,34541.490356


---

104. Rank users by total orders.

In [31]:
orders

,Field,Type,Null,Key,Default,Extra
0,order_id,varchar(36),NO,PRI,None,
1,user_id,varchar(36),YES,MUL,None,
2,order_date,datetime,YES,MUL,None,
3,status,text,YES,,None,


In [56]:
query_104 = """
    SELECT
        o.user_id,
        COUNT(o.order_id) order_count,
        RANK() OVER(ORDER BY COUNT(o.order_id) DESC) user_rank
    FROM
        orders o
    GROUP BY
        o.user_id
    LIMIT 200;
"""

In [57]:
show(query_104)

,user_id,order_count,user_rank
0,0be50a7b-2b5d-4021-9a0a-5e0ecd209a8d,13,1
1,33631649-9f9e-4db6-a3fd-4c0b6f5660fc,12,2
2,f11449d8-d778-415a-8bcc-0b3be1851f4e,12,2
3,51b4aa6c-15de-4d23-96e9-7548a7452ca9,12,2
4,72869271-c10d-4131-b7a3-624426a05161,11,5
...,...,...,...
195,e0db72f2-e36e-4604-a710-3e1075c2aca8,9,88
196,e1f9e14a-e447-4fd7-9629-1b600ae50363,9,88
197,e212d9fd-695d-4b30-b7dd-eecebb97b9b3,9,88
198,e29ceadb-d508-4c0f-8108-08cee87bcafd,9,88


----

105. Dense rank products per category.

In [58]:
products

,Field,Type,Null,Key,Default,Extra
0,product_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,category,text,YES,,None,
3,price,float,YES,MUL,None,
4,stock,int,YES,,None,


In [61]:
query_105 = """
    SELECT
        pr.product_id,
        DENSE_RANK() OVER(PARTITION BY pr.category ORDER BY pr.price DESC) product_rank
    FROM
        products pr
    LIMIT 20;
"""

In [62]:
show(query_105)

,product_id,product_rank
0,1be71191-46b0-4a04-8b72-0841a922fc25,1
1,0f4b500f-d323-407a-a540-b913e6dac977,2
2,f61fe562-9026-4c84-826e-7862bb4fed9e,3
3,4aee4880-b5ff-49e9-b819-9d29a24a0aed,4
4,c3ca5253-d70c-43fa-8d10-24c4d6314920,5
5,14a6dfdd-1f1a-47b2-ab5f-a1650c349db3,6
6,ff9b5bbf-a64b-4452-8e90-e78bbb4547bc,7
7,ca47970a-28cd-4990-911b-08f6a12ea085,8
8,32fe86e8-95ad-4131-b05b-26642f0b8281,9
9,ef2dd0ea-2988-4af3-b411-c9551b5cc1ff,10


----

106. Lag payment amounts.

In [79]:
query_106 = """
    SELECT
        pa.amount,
        LAG (pa.amount, 1) OVER (ORDER BY pa.payment_date )AS previous_amount
    FROM
        payments pa
    LIMIT 20;
"""

In [80]:
show(query_106)

,amount,previous_amount
0,2126.13,NaN
1,2148.34,2126.13
2,1576.66,2148.34
3,4214.58,1576.66
4,4522.18,4214.58
5,4332.88,4522.18
6,4575.35,4332.88
7,4836.12,4575.35
8,3979.28,4836.12
9,2229.97,3979.28


---

107. Lead order dates.

In [83]:
query_107 = """
    SELECT
        o.order_date,
        LEAD(o.order_date) OVER (ORDER BY o.order_date) AS Next_day
    FROM
        orders o
    LIMIT 20;
"""

In [84]:
show(query_107)

,order_date,Next_day
0,2026-01-01 00:00:05,2026-01-01 00:00:08
1,2026-01-01 00:00:08,2026-01-01 00:00:25
2,2026-01-01 00:00:25,2026-01-01 00:00:33
3,2026-01-01 00:00:33,2026-01-01 00:00:45
4,2026-01-01 00:00:45,2026-01-01 00:00:46
5,2026-01-01 00:00:46,2026-01-01 00:00:50
6,2026-01-01 00:00:50,2026-01-01 00:01:25
7,2026-01-01 00:01:25,2026-01-01 00:02:04
8,2026-01-01 00:02:04,2026-01-01 00:03:15
9,2026-01-01 00:03:15,2026-01-01 00:03:25


---

108. Cumulative sum of quantity sold.

In [87]:
order_items

,Field,Type,Null,Key,Default,Extra
0,order_item_id,varchar(36),NO,PRI,None,
1,order_id,varchar(36),YES,MUL,None,
2,product_id,varchar(36),YES,MUL,None,
3,quantity,int,YES,,None,


In [90]:
query_108 = """
    SELECT
        or_it.order_item_id,
        or_it.quantity,
        SUM(or_it.quantity) OVER (ORDER BY or_it.order_item_id) AS cumulative_sum
    FROM
        order_items or_it
    LIMIT 20;
"""

In [91]:
show(query_108)

,order_item_id,quantity,cumulative_sum
0,00000dd0-97cc-44fd-9538-3efc117f8219,2,2
1,00003118-eb66-47bc-9dc0-12111686ec6e,5,7
2,00003653-a179-4879-839c-cd9cd72b9761,4,11
3,00004c49-1691-45c0-8cc2-30f068615d80,2,13
4,00005156-71cd-489b-8744-93ebfc52cda7,1,14
5,000057d7-9611-4752-bd12-9ecf708311ad,1,15
6,00005cde-675d-4d4d-99d9-4dfb109154fe,1,16
7,0000857a-57c6-490d-b283-852a93976f79,2,18
8,0000bd08-7b2e-43fd-90dc-1434fd5b30f6,2,20
9,0000dc26-ad65-4c51-822c-86049576d5a8,4,24


---

109. Rank payments by amount per method.

In [109]:
query_109 = """
    SELECT
        pa.payment_id,
        pa.amount,
        pa.payment_method,
        RANK() OVER (PARTITION BY pa.payment_method ORDER BY pa.payment_id) AS RANK_
    FROM
        payments pa
    LIMIT 20;
"""

In [110]:
show(query_109)

,payment_id,amount,payment_method,RANK_
0,0000ce28-6059-41fe-807d-5dc6c602779f,4987.84,card,1
1,0004480f-2f59-4027-a3c3-0734740d5b91,4066.11,card,2
2,0004ad84-7984-4215-b67b-12dd1a7e1e63,18.54,card,3
3,0004d4d6-e230-4366-8185-abda11367762,3815.61,card,4
4,00062d8b-a19f-4978-aa4a-383350cb767e,3833.65,card,5
5,00080bbb-bfb1-4414-ac99-2d76c54fe380,4780.93,card,6
6,00082e83-6ca6-48f9-b72c-136762c3e7f0,2729.87,card,7
7,0008df68-a573-4cbd-85e6-5235d8052814,791.46,card,8
8,00099048-8eae-4915-8568-68c3dc36ed42,2745.57,card,9
9,000a72f3-db25-477b-9244-c62a695f97ab,4367.97,card,10


---

110. Row number partitioned by category.

In [112]:
products

,Field,Type,Null,Key,Default,Extra
0,product_id,varchar(36),NO,PRI,None,
1,name,text,YES,,None,
2,category,text,YES,,None,
3,price,float,YES,MUL,None,
4,stock,int,YES,,None,


In [119]:
query_110 = """
    SELECT
        pr.product_id,
        pr.category,
        ROW_NUMBER() OVER(PARTITION BY pr.category ORDER BY pr.product_id) AS ROW_NUM
    FROM
        products pr
    LIMIT 20;
"""

In [120]:
show(query_110)

,product_id,category,ROW_NUM
0,001a274d-067e-4cc2-92f7-7521cb51ed43,Books,1
1,00247d5d-d296-4c67-8fa5-c5ff6c1be085,Books,2
2,004befbb-4eac-401c-b3d0-efb1d9420fad,Books,3
3,0063d789-675a-4c9b-adab-d2253d3fd37f,Books,4
4,0075018e-d816-4052-8d73-d2f017f524bc,Books,5
5,00969008-8bd6-4a53-9e97-c5f26645b793,Books,6
6,00a3e6bb-e209-4ca4-b7f2-c4fcaf6369be,Books,7
7,00a55813-4a9b-4b66-b286-12b8c9dd7982,Books,8
8,00c0fdd4-9e10-4127-a199-a93eb83aa51b,Books,9
9,00d1cdc9-93ef-4eca-b833-f2f046651b01,Books,10


---